# Soft-collection segmentation: auto loans, DPD 1-30

**Method in 10 lines**
1. `ovrd` (monthly max DPD per contract) is turned into a DPD panel and cut into **delinquency episodes**. Every entry into DPD 1+ is one observation, with `obs_date` = the first overdue day.
2. **Target** `bad`: the contract reaches 90+ DPD within 6 months after `obs_date`. Secondary targets: `self_cure30` (the episode never passes 30 DPD) and `roll31_60d` (31+ DPD within 60 days).
3. Every feature is computed **as of `obs_date`**: only events dated strictly before it. Each data source is a plug-in *feature block* `fb_*(obs, ctx)`.
4. Sources that start late (geo, vnpt, vneshpt, PKB events) feed the rules and overlays only, so the model does not learn time artefacts.
5. A **LightGBM PD model**, benchmarked against a logistic regression, is trained with a time split. Out-of-time (OOT) = the last 3 labelled months.
6. PD is cut into **risk grades G1-G5** at fixed cutpoints taken from the train-sample PD quantiles (40/65/85/95%).
7. A **rule overlay** takes priority: deceased, bankruptcy, exclusion lists, enforcement, skip/abroad, first-payment default (FPD) and fraud, gambling, collateral.
8. Everything else is split by **risk x ability to pay (capacity) x willingness/reachability (engagement)** into six treatment segments, S1-S6.
9. Segments are **backtested** on labelled history and profiled with behavioural clusters. The **current stock** (DPD 1-30 now) gets a segment, a treatment and its compliance dates.
10. Outputs: parquet and Excel files, charts and the model artefacts, all in `out/`.

**How to run** (pick one)
- **(a) Same kernel as your loading notebook.** Load the tables under their usual names (`contr`, `ovrd`, `ver_df_all_2024_2026`, `bankrupt`, `ocrm_call`, `vneshpt_ot`, `vnpt`, `excl`, `vneshpt`, `geo`, `dm`, `dm_rej`, optionally `hes_pkb_agg`), then run this notebook in that kernel with `%run -i soft_collection_segmentation.ipynb`, or paste the cells. Tables are picked up from `globals()` and are never modified.
- **(b) Parquet files.** Set `CONFIG['DATA_DIR']` and the file names in `CONFIG['FILES']` (globs allowed). Only the needed columns and the population's keys are read.
- **(c) DEMO.** If no tables are found, `MODE='auto'` falls back to the synthetic generator `demo_data.py`, which has the same schema and fake people. `MODE='demo'` forces it.

Only `contr` and `ovrd` are required. A missing optional source only switches its block off.

Written for Python 3.13/3.14 and pandas 3: `str` dtype, Copy-on-Write, no chained assignment.

## 1. Configuration
Every knob lives in one dict, `CONFIG`. The test runner can override `MODE`, `DROP_TABLES`, `OUT_DIR`, `DATA_DIR`, `DEMO_N_CONTRACTS`, `USE_LIGHTGBM` and `STRICT_BLOCKS` through `SEG_*` environment variables. Without them, nothing changes.

In [ ]:
import os

CONFIG = {
    # ---- where the data comes from ---------------------------------------------------------------
    'MODE': 'auto',            # 'auto': globals() -> parquet -> DEMO | 'real': globals()/parquet only | 'demo'
    'DATA_DIR': 'data',        # folder with parquet extracts (way (b)), relative to this notebook
    'FILES': {                 # table -> parquet file name, glob or list  (PLACEHOLDERS: edit to your names)
        'contr': 'contr_auto042024_032026.parquet',
        'ovrd': 'ovrd.parquet',
        'ver_df_all_2024_2026': 'ver_df_all_2024_2026.parquet',
        'bankrupt': 'bankrupt.parquet',
        'ocrm_call': 'ocrm_call.parquet',
        'vneshpt_ot': 'vneshpt_ot.parquet',
        'vnpt': 'vnpt.parquet',
        'excl': 'excl.parquet',
        'vneshpt': 'vneshpt.parquet',
        'geo': 'geo.parquet',
        'dm': 'dm.parquet',
        'dm_rej': 'dm_*_rejected.parquet',      # e.g. one file per year
    },
    'PKB_AGG_FILE': 'hes_pkb_agg.parquet',      # optional: output of sql/hes_pkb_monitoring_agg.sql (in DATA_DIR)
    'DROP_TABLES': [],         # switch sources off (also used by the robustness test)
    'DEMO_N_CONTRACTS': 8000,  # synthetic portfolio size in DEMO mode (8000: ~750 OOT episodes, ~25 s, ~2 GB RAM)
    'STRICT_BLOCKS': False,    # True: a failing optional block raises; False: warn and skip it
    # ---- episodes and target ---------------------------------------------------------------------
    'SOFT_DPD_MIN': 1, 'SOFT_DPD_MAX': 30,       # soft-collection window (current stock)
    'BAD_DPD': 90, 'HORIZON_MONTHS': 6,          # bad = reaches 90+ DPD within 6 months after obs_date
    'ROLL_DPD': 31, 'ROLL_DAYS': 60,             # fast proxy target: 31+ DPD within 60 days
    'OBS_OFFSET_DAYS': 0,      # obs_date = entry_date + offset (0 = score on the first overdue day)
    'EPISODE_GAP_DAYS': 5,     # implied start dates further apart than this -> new episode
    'STOCK_ONLY_AT_CUT': True, # current stock = rows still overdue on the data cut (act_date == cut)
    'PROFILE_LAG_DAYS': 0,     # ver_df snapshot must be dated < obs_date - lag (raise if snapshots land late)
    # ---- model -----------------------------------------------------------------------------------
    'OOT_MONTHS': 3, 'VALID_SHARE': 0.2,
    'GRADE_QUANTILES': [0.40, 0.65, 0.85, 0.95],  # train-PD quantiles -> cutpoints G1|G2|G3|G4|G5
    'RANDOM_STATE': 42,
    'MODEL_BLOCKS': 'auto',    # 'auto' (rule in section 6) or an explicit list of block names
    'MODEL_EXCLUDE': ['pr_snapshot_age_days',     # time artefact (snapshot cadence changes over time)
                      'ct_term_months', 'ct_remaining_months'],  # from contr.end_date = extraction-time value
                                                 # (changes on restructuring) -> leakage risk
    'PSI_DROP_THRESHOLD': 0.25, 'MAX_NULL_SHARE': 0.98,
    'TOP_N_CATEGORIES': 15,
    'USE_LIGHTGBM': True,      # False or package missing -> sklearn HistGradientBoosting
    'LR_TOP_N': 25,
    # ---- segmentation ----------------------------------------------------------------------------
    'KZ_BBOX': (40.5, 55.5, 46.4, 87.4),         # lat_min, lat_max, lon_min, lon_max
    'CAPACITY_RATIO_MIN': 1.5, 'DEPO_COVER_MIN': 1.0, 'CONTACT_RATE_ENGAGED': 0.5,
    'CHALLENGER_SHARE': 0.15,  # share of every segment routed to the challenger treatment (contract hash)
    'R02_PKB_COURT': True,     # PKB 'court' status (may concern ANOTHER creditor's loan) also triggers R02 - Legal to decide
    'CLUSTER_K_RANGE': (4, 8), 'CLUSTER_SAMPLE': 20000,
    # ---- regulatory constants (research memo, 2026; Legal to confirm items marked [verify]) --------
    'COMPLIANCE': {
        'NOTIFY_DAYS': 10, 'RESTRUCT_APPLY_DAYS': 30, 'RESTRUCT_REVIEW_DAYS': 15,
        'CALLS': {'weekday': (8, 21, 3), 'weekend': (9, 19, 2)},   # (from hour, to hour, max calls per day)
        'VISITS': (1, 3),                                          # max per day, per week (weekdays only)
        'PM_2026': 50851, 'DEBIT_MAX_SHARE': 0.5, 'DEBIT_PROTECTED': 2 * 50851,
        'MRP_2026': 4325, 'OOC_BANKRUPTCY_MAX': 1600 * 4325,
        'NO_NEW_LOAN_DPD': 30, 'COLLECTOR_HANDOVER_DPD': 90, 'ASSIGNMENT_BAN_UNTIL': '2027-05-01',
    },
    # ---- outputs ---------------------------------------------------------------------------------
    'OUT_DIR': 'out',
}

# Overrides from the environment (used by the test runner; harmless otherwise)
_env = os.environ
if _env.get('SEG_MODE'):
    CONFIG['MODE'] = _env['SEG_MODE'].strip().lower()
if _env.get('SEG_DROP_TABLES'):
    CONFIG['DROP_TABLES'] = [t.strip() for t in _env['SEG_DROP_TABLES'].split(',') if t.strip()]
for _k in ('OUT_DIR', 'DATA_DIR'):
    if _env.get(f'SEG_{_k}'):
        CONFIG[_k] = _env[f'SEG_{_k}']
if _env.get('SEG_DEMO_N_CONTRACTS'):
    CONFIG['DEMO_N_CONTRACTS'] = int(_env['SEG_DEMO_N_CONTRACTS'])
for _k in ('USE_LIGHTGBM', 'STRICT_BLOCKS'):
    if _env.get(f'SEG_{_k}'):
        CONFIG[_k] = _env[f'SEG_{_k}'].strip().lower() in ('1', 'true', 'yes', 'y')
assert CONFIG['MODE'] in ('auto', 'demo', 'real'), CONFIG['MODE']
print({k: CONFIG[k] for k in ('MODE', 'DATA_DIR', 'OUT_DIR', 'DROP_TABLES', 'USE_LIGHTGBM', 'STRICT_BLOCKS')})

## 2. Imports and helpers
The helpers are safe under pandas 3 (`str` dtype with NaN as missing, Copy-on-Write):
- `norm_iin`, `norm_contract`, `norm_ref` normalise keys. They run on the unique values only, so they are fast on tables with 10^7 rows. IINs keep or regain their leading zeros (`zfill(12)`).
- `to_dt` returns `datetime64[us]` and turns the 9999-12-31 "open" sentinel into NaT. It never casts to `[ns]`, where that date overflows.
- `flag01` reads string flags (`'1'`, `'1.0'`, `'Y'`, `'Да'`, `'TRUE'` ...) as 0/1.
- `load_table` finds each table in this priority order: (1) a DataFrame in `globals()`, (2) parquet with column projection and key push-down, (3) DEMO.

In [ ]:
import glob, hashlib, json, math, pickle, platform, re, sys, time, traceback
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import sklearn
from sklearn.base import clone
from sklearn.cluster import KMeans
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, silhouette_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import QuantileTransformer, StandardScaler
from IPython.display import Markdown, display

try:
    import lightgbm as lgb
    HAS_LGB = True
except Exception:          # not installed or a broken wheel -> HistGradientBoosting fallback
    lgb, HAS_LGB = None, False

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

OUT = Path(CONFIG['OUT_DIR'])
for _d in (OUT, OUT / 'charts', OUT / 'model'):
    _d.mkdir(parents=True, exist_ok=True)

print(f"python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__} | "
      f"scikit-learn {sklearn.__version__} | lightgbm {lgb.__version__ if HAS_LGB else 'n/a'} | "
      f"matplotlib {matplotlib.__version__}")
if not pd.__version__.startswith('3'):
    print('NOTE: written for pandas 3.x (str dtype, Copy-on-Write).')

In [ ]:
T0 = time.perf_counter()


def log(msg):
    print(f'[{time.perf_counter() - T0:6.1f}s] {msg}', flush=True)


@contextmanager
def timer(name):
    t = time.perf_counter()
    yield
    log(f'{name}: {time.perf_counter() - t:.1f}s')


def show(df, title=None):
    """Display a compact table under a bold title."""
    if title:
        display(Markdown(f'**{title}**'))
    display(df)


def _via_uniques(s, fn):
    """Apply a vectorised string normaliser to the unique values only (fast on 10^7-row columns)."""
    s = pd.Series(s)
    codes, uniq = pd.factorize(s, use_na_sentinel=True)
    out = np.full(len(s), np.nan, dtype=object)
    if len(uniq):
        normed = fn(pd.Series(np.asarray(uniq, dtype=object), dtype='str'))
        normed = normed.to_numpy(dtype=object, na_value=np.nan)
        ok = codes >= 0
        out[ok] = normed[codes[ok]]
    return pd.Series(out, index=s.index, dtype='str')


def _clean_digits(u):
    return u.str.strip().str.replace(r'\.0+$', '', regex=True).str.replace(r'\D', '', regex=True)


def norm_iin(s):
    """IIN/BIN -> 12-digit string with leading zeros restored; anything else -> NaN."""
    def f(u):
        x = _clean_digits(u)
        x = x.where(x.str.len().between(1, 12)).str.zfill(12)
        return x.where(x != '000000000000')
    return _via_uniques(s, f)


def norm_contract(s):
    """Contract / account number -> digits-only string ('.0' float artefacts removed)."""
    def f(u):
        x = _clean_digits(u)
        return x.where(x.str.len() > 0)
    return _via_uniques(s, f)


def norm_ref(s):
    """Process reference (e.g. DGA000000000001) -> stripped upper-case string."""
    def f(u):
        x = u.str.strip().str.upper()
        return x.where(x.str.len() > 0)
    return _via_uniques(s, f)


FAR_FUTURE = pd.Timestamp('2100-01-01')
_NA_TXT = ['', 'NaT', 'nan', 'NaN', 'None', 'NULL', 'null']


def to_dt(s, max_ts=FAR_FUTURE):
    """Any date-like column -> datetime64[us]. Strings: ISO first, then mixed / day-first.
    Sentinels >= 2100 (9999-12-31 = 'still open') -> NaT. Never casts to [ns] (9999-12-31 overflows ns)."""
    s = pd.Series(s)
    if isinstance(s.dtype, pd.DatetimeTZDtype):
        s = s.dt.tz_localize(None)
    if pd.api.types.is_datetime64_dtype(s):
        out = s
    else:
        txt = s.astype('str').str.strip()
        out = pd.to_datetime(txt, errors='coerce', format='ISO8601')
        miss = out.isna() & txt.notna() & ~txt.isin(_NA_TXT)
        if miss.any():
            alt = pd.to_datetime(txt[miss], errors='coerce', format='mixed', dayfirst=True)
            out = out.where(out < max_ts).astype('datetime64[us]').fillna(alt.where(alt < max_ts).astype('datetime64[us]'))
    out = out.where(out < max_ts)
    return out.astype('datetime64[us]')


def coalesce(*series):
    out = series[0]
    for s in series[1:]:
        out = out.fillna(s)
    return out


def text_cols(df):
    """Text-like columns: object, str (pandas 3 default), string[pyarrow] or category."""
    return [c for c in df.columns
            if pd.api.types.is_object_dtype(df[c]) or pd.api.types.is_string_dtype(df[c])
            or isinstance(df[c].dtype, pd.CategoricalDtype)]


_TRUE = {'1', '1.0', 'Y', 'YES', 'ДА', 'TRUE', 'T'}
_FALSE = {'0', '0.0', 'N', 'NO', 'НЕТ', 'FALSE', 'F'}


def flag01(s):
    """Robust 0/1 parser for string flags ('1', '1.0', 'Y', 'Да', 'TRUE' -> 1; '0', 'N', 'Нет' -> 0; else NaN)."""
    s = pd.Series(s)
    if pd.api.types.is_bool_dtype(s):
        return s.astype('float32')
    if pd.api.types.is_numeric_dtype(s):
        x = s.astype('float64')
        return x.where(x.isin([0.0, 1.0])).astype('float32')
    u = s.astype('str').str.strip().str.upper()
    out = np.where(u.isin(_TRUE), 1.0, np.where(u.isin(_FALSE), 0.0, np.nan))
    return pd.Series(out, index=s.index, dtype='float32')


def to_float32(df):
    """Memory-light downcast of numeric (and bool) columns to float32."""
    num = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    return df.astype({c: 'float32' for c in num}) if num else df


def col(df, name, default=np.nan):
    """df[name] if present, else a constant float Series (a source or block may be missing)."""
    return df[name] if name in df.columns else pd.Series(default, index=df.index, dtype='float64')


def as_cat(s, upper=True):
    """Text category: str, stripped (upper-cased), '.0' artefacts removed; NaN stays NaN."""
    x = pd.Series(s).astype('str').str.strip()
    if upper:
        x = x.str.upper()
    x = x.str.replace(r'\.0$', '', regex=True)
    return x.where(~x.isin(_NA_TXT + ['NAN', 'NONE', 'NAT']))


def psi(expected, actual, bins=10, eps=1e-4):
    """Population stability index of `actual` vs `expected` (numeric: expected-decile bins, text:
    categories). Missing values form their own bin."""
    e, a = pd.Series(expected), pd.Series(actual)
    if len(e) == 0 or len(a) == 0:
        return np.nan
    if pd.api.types.is_numeric_dtype(e) and pd.api.types.is_numeric_dtype(a):
        ev, av = e.to_numpy(dtype='float64'), a.to_numpy(dtype='float64')
        fin = ev[np.isfinite(ev)]
        edges = np.unique(np.quantile(fin, np.linspace(0, 1, bins + 1)[1:-1])) if len(fin) else np.array([])

        def dist(v):
            idx = np.where(np.isnan(v), -1, np.searchsorted(edges, v, side='right'))
            return pd.Series(idx).value_counts(normalize=True)
        pe, pa = dist(ev), dist(av)
    else:
        pe = e.astype('str').fillna('<NA>').value_counts(normalize=True)
        pa = a.astype('str').fillna('<NA>').value_counts(normalize=True)
    keys = pe.index.union(pa.index)
    pe = pe.reindex(keys, fill_value=0).to_numpy() + eps
    pa = pa.reindex(keys, fill_value=0).to_numpy() + eps
    return float(np.sum((pa - pe) * np.log(pa / pe)))


def perf(y, p):
    """n, bad rate, AUC, Gini, KS."""
    y, p = np.asarray(y, dtype='float64'), np.asarray(p, dtype='float64')
    out = dict(n=len(y), bad_rate=float(y.mean()) if len(y) else np.nan, auc=np.nan, gini=np.nan, ks=np.nan)
    if len(np.unique(y)) == 2:
        auc = roc_auc_score(y, p)
        fpr, tpr, _ = roc_curve(y, p)
        out.update(auc=auc, gini=2 * auc - 1, ks=float(np.max(tpr - fpr)))
    return out


def wilson(k, n, z=1.96):
    """Wilson 95% confidence interval for a proportion k/n."""
    k, n = np.asarray(k, dtype='float64'), np.asarray(n, dtype='float64')
    with np.errstate(divide='ignore', invalid='ignore'):
        p = k / n
        den = 1 + z ** 2 / n
        c = (p + z ** 2 / (2 * n)) / den
        h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / den
    return c - h, c + h


def hash_share(keys, salt='soft-collection-cc'):
    """Deterministic pseudo-random number in [0, 1) per key (champion/challenger routing)."""
    return np.array([int(hashlib.md5(f'{salt}|{k}'.encode()).hexdigest()[:12], 16) / 16 ** 12 for k in keys])

In [ ]:
REQUIRED_TABLES = ('contr', 'ovrd')
LOAD_LOG = []
_DEMO = {}


def _demo_tables():
    """Synthetic tables with the bank's schema, from demo_data.py next to this notebook."""
    if 'tables' not in _DEMO:
        here = str(Path.cwd())
        if here not in sys.path:
            sys.path.insert(0, here)
        try:
            from demo_data import make_demo_tables
        except ImportError as e:
            raise ImportError('DEMO mode needs demo_data.py in the notebook folder') from e
        with timer(f"DEMO data generated (n_contracts={CONFIG['DEMO_N_CONTRACTS']})"):
            _DEMO['tables'] = make_demo_tables(n_contracts=CONFIG['DEMO_N_CONTRACTS'], seed=CONFIG['RANDOM_STATE'])
    return _DEMO['tables']


def _parquet_paths(name):
    spec = CONFIG['PKB_AGG_FILE'] if name == 'hes_pkb_agg' else CONFIG['FILES'].get(name)
    if not spec:
        return []
    paths = []
    for sp in ([spec] if isinstance(spec, (str, Path)) else list(spec)):
        p = Path(sp) if Path(sp).is_absolute() else Path(CONFIG['DATA_DIR']) / sp
        if any(ch in str(p) for ch in '*?['):
            paths += [Path(x) for x in sorted(glob.glob(str(p)))]
        elif p.exists():
            paths.append(p)
    return paths


def _user_df(name):
    obj = globals().get(name)
    return obj if isinstance(obj, pd.DataFrame) else None


def resolve_mode():
    """'real' if the required tables are in globals() or on disk, else 'demo' (MODE='auto')."""
    if CONFIG['MODE'] == 'demo':
        return 'demo'
    found = {t: 'globals' if _user_df(t) is not None else ('parquet' if _parquet_paths(t) else None)
             for t in REQUIRED_TABLES}
    if all(found.values()):
        return 'real'
    if CONFIG['MODE'] == 'real':
        raise RuntimeError(f"MODE='real' but {[t for t, v in found.items() if not v]} not found in globals() "
                           f"nor in DATA_DIR={CONFIG['DATA_DIR']!r}")
    return 'demo'


def _key_variants(keys, kind):
    ks = {k for k in keys if isinstance(k, str) and k}
    out = set(ks)
    if kind == 'iin':
        out |= {k.lstrip('0') for k in ks}        # IINs that lost their leading zeros
    return out | {k + '.0' for k in out}         # float artefacts ('900101300001.0')


def _numeric_keys(keys):
    v = pd.to_numeric(pd.Series(sorted(k for k in keys if isinstance(k, str) and k), dtype='str'), errors='coerce')
    return v.dropna().unique().tolist()


_NORMALISERS = {'iin': norm_iin, 'contract': norm_contract, 'ref': norm_ref}


def _key_mask(s, keys, kind):
    """Boolean mask: the raw key value is one of the (normalised) population keys."""
    if pd.api.types.is_numeric_dtype(s):
        return s.isin(_numeric_keys(keys)).to_numpy()
    m = s.isin(_key_variants(keys, kind)).to_numpy()
    if not m.any() and len(s):          # unusual raw format -> normalise the whole column once
        m = _NORMALISERS[kind](s).isin(keys).to_numpy()
    return m


def _read_parquet(paths, columns=None, filters=None):
    """Read parquet file(s) with column projection and key push-down (rows are re-checked in pandas)."""
    import pyarrow as pa
    import pyarrow.parquet as pq
    is_str = lambda t: pa.types.is_string(t) or pa.types.is_large_string(t) or getattr(pa.types, 'is_string_view', lambda _: False)(t)
    parts = []
    for p in paths:
        schema = pq.read_schema(p)
        cols = [c for c in columns if c in schema.names] if columns else None
        dnf = []
        for kcol, kind, keys in filters or []:
            if kcol not in schema.names:
                continue
            typ = schema.field(kcol).type
            if pa.types.is_integer(typ):
                vals = [int(v) for v in _numeric_keys(keys)] or [-1]
            elif pa.types.is_floating(typ):
                vals = [float(v) for v in _numeric_keys(keys)] or [-1.0]
            elif is_str(typ):
                vals = sorted(_key_variants(keys, kind)) or ['<none>']
            else:                                 # dictionary / other types: no push-down for this file
                dnf = None
                break
            dnf.append([(kcol, 'in', vals)])
        parts.append(pq.read_table(p, columns=cols, filters=dnf or None).to_pandas())
    return parts[0] if len(parts) == 1 else pd.concat(parts, ignore_index=True)


def load_table(name, columns=None, iin_filter=None, iin_col=None, key_filters=None):
    """Load one table. Priority: (1) a DataFrame named `name` in globals() (your kernel), (2) parquet
    DATA_DIR/FILES[name], (3) the DEMO generator (only when RUN_MODE == 'demo').
    Rows are restricted to the population: `iin_filter` on `iin_col` and/or `key_filters`
    = [(column, kind, keys), ...] (OR-combined; kind in iin | contract | ref). Missing columns are ignored.
    Returns None (with a warning) if the table is unavailable or dropped; its block is then skipped."""
    t0 = time.perf_counter()
    filters = list(key_filters or [])
    if iin_filter is not None and iin_col:
        filters.append((iin_col, 'iin', set(iin_filter)))
    rec = dict(table=name, source=None, rows_raw=np.nan, rows_loaded=0, n_cols=0, missing_cols='', seconds=0.0)
    df = None
    if name in CONFIG['DROP_TABLES']:
        rec['source'] = 'dropped (CONFIG)'
    elif RUN_MODE == 'real':
        g = _user_df(name)
        if g is not None:
            df, rec['source'], rec['rows_raw'] = g, 'globals', len(g)
        elif (paths := _parquet_paths(name)):
            import pyarrow.parquet as pq
            rec['rows_raw'] = sum(pq.read_metadata(p).num_rows for p in paths)
            df, rec['source'] = _read_parquet(paths, columns, filters), f'parquet x{len(paths)}'
    else:
        demo = _demo_tables()
        if name in demo:
            df, rec['source'], rec['rows_raw'] = demo[name], 'demo', len(demo[name])
    if df is None:
        rec['source'] = rec['source'] or 'not found'
        LOAD_LOG.append(rec)
        print(f"WARN  {name}: {rec['source']} -> dependent block skipped")
        return None
    keep = list(df.columns) if columns is None else [c for c in columns if c in df.columns]
    rec['missing_cols'] = ', '.join(c for c in (columns or []) if c not in df.columns)[:120]
    active = [(c, k, v) for c, k, v in filters if c in df.columns]
    if filters and not active:
        print(f'WARN  {name}: none of the key columns {[c for c, _, _ in filters]} found -> no population filter')
    if active:
        mask = np.zeros(len(df), dtype=bool)
        for c, kind, keys in active:
            mask |= _key_mask(df[c], keys, kind)
        df = df.loc[mask, keep]
    else:
        df = df[keep]
    df = df.reset_index(drop=True)            # a new object: the caller's DataFrame is never modified
    rec.update(rows_loaded=len(df), n_cols=df.shape[1], seconds=round(time.perf_counter() - t0, 2))
    LOAD_LOG.append(rec)
    return df


RUN_MODE = resolve_mode()
log(f"RUN_MODE = {RUN_MODE!r} (CONFIG['MODE'] = {CONFIG['MODE']!r}); dropped tables: {CONFIG['DROP_TABLES'] or 'none'}")

## 3. Data load, normalisation and data-quality report
1. **Required tables.** `contr` is normalised and de-duplicated: the same contract can appear twice with different `end_date`, so the row with the latest `open_date`, then the latest `end_date`, is kept. `ovrd` is restricted to contracts in `contr` with DPD > 0.
2. **Population** = contracts that ever appear in `ovrd` with DPD > 0, plus their clients and process refs. Every optional table is loaded **only for these keys and only with the needed columns**. At real scale this is what keeps `geo` (37M rows) and `vneshpt` (33M rows) manageable.
3. **Data-quality report**: rows, key nulls, duplicates and date ranges for each table.

`close_date`, `status_name` and `restructing_fl` in `contr` are extraction-time values. They are never model features.

In [ ]:
contr_raw = load_table('contr')
if contr_raw is None:
    raise RuntimeError('contr is required (globals / parquet / demo)')
_c = contr_raw.assign(contract_number=norm_contract(contr_raw['contract_number']),
                      iin=norm_iin(contr_raw['iin']),
                      process_ref=norm_ref(col(contr_raw, 'process_ref')))
for _dcol in ('open_date', 'end_date', 'close_date'):
    if _dcol in _c.columns:
        _c[_dcol] = to_dt(_c[_dcol])
_n_raw = len(_c)
contracts = (_c.dropna(subset=['contract_number'])
               .sort_values(['contract_number', 'open_date', 'end_date'], na_position='first')
               .drop_duplicates('contract_number', keep='last').reset_index(drop=True))
_raw_iin = contr_raw['iin'].astype('str').str.strip().str.replace(r'\.0+$', '', regex=True)
contr_summary = pd.Series({
    'contr rows (raw)': _n_raw,
    'duplicate contract rows dropped': _n_raw - len(contracts),
    'contracts': len(contracts),
    'clients (IIN)': contracts['iin'].nunique(),
    'invalid / missing IIN': int(contracts['iin'].isna().sum()),
    'IINs that needed leading zeros restored': int((_raw_iin.str.len() < 12).sum()),
    'process_ref shared by >1 contract': int(contracts['process_ref'].dropna().duplicated().sum()),
    'open_date range': f"{contracts['open_date'].min():%Y-%m-%d} .. {contracts['open_date'].max():%Y-%m-%d}",
}, name='value', dtype=object)

LEAKAGE_COLS = ('close_date', 'status_name', 'restructing_fl')
if 'status_name' in contracts.columns:
    _st = contracts['status_name'].value_counts(dropna=False).rename('contracts').to_frame()
    _st['share'] = _st['contracts'] / len(contracts)
    show(_st, 'contr.status_name (status at extraction time)')
    if set(contracts['status_name'].dropna().unique()) <= {'Актуален'}:
        print('WARNING: contr holds only currently active (Актуален) contracts -> closed / written-off / sold '
              'contracts are missing: survivorship bias (the worst outcomes are under-represented). '
              'Please re-extract contr including closed contracts of the same cohorts.')
    else:
        _na = (contracts['status_name'] != 'Актуален').mean()
        print(f'contr includes non-active statuses ({_na:.1%} of contracts). Check that ALL closed / written-off '
              'contracts of the cohorts are there; otherwise the history suffers from survivorship bias.')
print('Leakage guard: close_date / status_name / restructing_fl are extraction-time values -> never model features.')

In [ ]:
ovrd_raw = load_table('ovrd')
if ovrd_raw is None:
    raise RuntimeError('ovrd is required (it is the target source)')
_o = ovrd_raw.assign(contract_number=norm_contract(ovrd_raw['contract_number']),
                     act_date=to_dt(ovrd_raw['act_date']),
                     dpd=pd.to_numeric(ovrd_raw['ovd_days_cnt'], errors='coerce'))
_rm = to_dt(col(ovrd_raw, 'rep_month')).fillna(_o['act_date'])
_o['rep_month'] = (_rm - pd.to_timedelta(_rm.dt.day - 1, unit='D')).dt.normalize().astype('datetime64[us]')
_known = _o['contract_number'].isin(set(contracts['contract_number']))
ovrd_stats = pd.Series({
    'ovrd rows (raw)': len(_o),
    'rows of contracts not in contr (other products, dropped)': int((~_known).sum()),
    'rows with DPD <= 0 / missing (dropped)': int((~(_o['dpd'] > 0)).sum()),
    'rows with unparseable act_date (dropped)': int(_o['act_date'].isna().sum()),
}, name='value', dtype=object)
ovrd_clean = _o[_known & (_o['dpd'] > 0) & _o['act_date'].notna()].reset_index(drop=True)

POP_CONTRACTS = set(ovrd_clean['contract_number'])
_pop = contracts[contracts['contract_number'].isin(POP_CONTRACTS)]
POP_IINS, POP_REFS = set(_pop['iin'].dropna()), set(_pop['process_ref'].dropna())
POP_KEYS = {'iin': POP_IINS, 'contract': POP_CONTRACTS, 'ref': POP_REFS}
ovrd_stats['population: ever-delinquent contracts'] = len(POP_CONTRACTS)
ovrd_stats['population: clients'] = len(POP_IINS)
ovrd_stats['share of portfolio ever delinquent'] = f'{len(POP_CONTRACTS) / len(contracts):.1%}'
show(pd.concat([contr_summary, ovrd_stats]).to_frame(), 'Required tables')

In [ ]:
# ---- what to read from each optional source: columns and population keys (OR-combined) ----------
TABLE_COLUMNS = {
    'ver_df_all_2024_2026': None,     # all 203 columns (171 numeric are features)
    'bankrupt': ['iin', 'processcreate', 'date_value', 'date_change', 'bankruptcytype', 'statementstatus',
                 'request_amount'],
    'ocrm_call': ['callid', 'num_dbz', 'createdon_date', 'date_change1', 'date_value1', 'talktime', 'result_predl',
                  'call_comment', 'tnumb_to', 'tsptp_date', 'activityid', 'act_creat_date', 'fact_zvr_date',
                  'date_value', 'activ_sost', 'activ_result', 'activ_result_dtl', 'company_sc', 'ocrm_tsdebt_id'],
    'vneshpt_ot': ['req_id', 'iin_bin', 'recip_client_iin', 'is_actual', 'date_change', 'accept_date', 'close_date',
                   'amount_sum', 'opertype_name', 'descr', 'k2_requirement_group_id', 'k2_req_status_id'],
    'vnpt': ['req_id', 'dogovor', 'doc_num', 'iin_bin', 'recip_client_iin', 'is_actual', 'date_change',
             'accept_date', 'close_date', 'amount_sum', 'k2_req_status_id'],
    'excl': ['iin_bin', 'is_active_fl', 'exclude_title', 'is_actual', 'updated_date'],
    'vneshpt': ['РЕФЕРЕНС', 'ИИН', 'ext_dea_code', 'longname', 'status_code', 'status_name', 'Причина',
                'created_date', 'date_change', 'debt'],
    'geo': ['client_iin', 'event_datetime', 'date_value', 'action', 'latitude', 'longitude', 'device_id',
            'device_name', 'ip_address', 'app_version_name'],
    'dm': None, 'dm_rej': None,       # 159 columns, auto-profiled
    'hes_pkb_agg': None,
}
TABLE_KEYS = {
    'ver_df_all_2024_2026': [('IIN', 'iin'), ('iin', 'iin')],
    'bankrupt': [('iin', 'iin')],
    'ocrm_call': [('num_dbz', 'contract')],
    'vneshpt_ot': [('iin_bin', 'iin'), ('recip_client_iin', 'iin')],
    'vnpt': [('dogovor', 'contract'), ('iin_bin', 'iin')],
    'excl': [('iin_bin', 'iin')],
    'vneshpt': [('ИИН', 'iin'), ('ext_dea_code', 'contract')],
    'geo': [('client_iin', 'iin')],
    'dm': [('process_ref', 'ref')],
    'dm_rej': [('iin', 'iin'), ('IIN', 'iin'), ('client_iin', 'iin'), ('iin_bin', 'iin')],
    'hes_pkb_agg': [('iin', 'iin')],
}

# ---- per-source preparation (keys normalised, dates parsed, duplicates removed) -------------------
DQ_NOTES = {}


def prep_ver(v):
    v = v.rename(columns={'IIN': 'iin'})
    v = v.assign(iin=norm_iin(v['iin']), DATE_VALUE=to_dt(v['DATE_VALUE']))
    if 'DOC_EXPIRY_DATE' in v.columns:
        v['DOC_EXPIRY_DATE'] = to_dt(v['DOC_EXPIRY_DATE'])
    n0 = len(v)
    v = v.dropna(subset=['iin', 'DATE_VALUE']).drop_duplicates(['iin', 'DATE_VALUE'], keep='last')
    DQ_NOTES['ver_df_all_2024_2026'] = f'{n0 - len(v):,} rows dropped (missing key / duplicate iin x DATE_VALUE)'
    return to_float32(v).reset_index(drop=True)


def prep_bankrupt(b):
    b = b.assign(iin=norm_iin(b['iin']),
                 event_date=coalesce(to_dt(col(b, 'processcreate')), to_dt(col(b, 'date_value')),
                                     to_dt(col(b, 'date_change'))),
                 refused=(col(b, 'statementstatus').astype('str').str.contains('otkaz|отказ', case=False, na=False)
                          if 'statementstatus' in b.columns else pd.Series(True, index=b.index)).astype('float32'))
    types = b['bankruptcytype'].value_counts().to_dict() if 'bankruptcytype' in b.columns else {}
    DQ_NOTES['bankrupt'] = f'types {types}; refused share {b["refused"].mean():.0%}'
    return b.dropna(subset=['iin', 'event_date']).reset_index(drop=True)


def prep_ocrm(o):
    o = o.assign(contract_number=norm_contract(o['num_dbz']))
    empty = [c for c in o.columns if o[c].isna().all()]
    o = o.drop(columns=empty)                 # all-null technical columns
    calls = acts = None
    if 'callid' in o.columns:
        calls = o.dropna(subset=['callid']).drop_duplicates('callid')
        calls = calls.assign(call_ts=coalesce(to_dt(col(calls, 'createdon_date')), to_dt(col(calls, 'date_change1')),
                                              to_dt(col(calls, 'date_value1'))),
                             talktime=pd.to_numeric(col(calls, 'talktime'), errors='coerce'),
                             tsptp_date=to_dt(col(calls, 'tsptp_date')))
        keep = ['contract_number', 'callid', 'call_ts', 'talktime', 'result_predl', 'tnumb_to', 'tsptp_date',
                'call_comment']
        calls = calls[[c for c in keep if c in calls.columns]].dropna(subset=['call_ts', 'contract_number'])
    if 'activityid' in o.columns:
        acts = o.dropna(subset=['activityid']).drop_duplicates('activityid')
        acts = acts.assign(act_ts=coalesce(to_dt(col(acts, 'act_creat_date')), to_dt(col(acts, 'fact_zvr_date')),
                                           to_dt(col(acts, 'date_value'))))
        keep = ['contract_number', 'activityid', 'act_ts', 'activ_sost', 'activ_result', 'activ_result_dtl',
                'ocrm_tsdebt_id', 'company_sc']
        acts = acts[[c for c in keep if c in acts.columns]].dropna(subset=['act_ts', 'contract_number'])
    n_c = 0 if calls is None else len(calls)
    n_a = 0 if acts is None else len(acts)
    DQ_NOTES['ocrm_call'] = (f'calls x activities join: {len(o):,} rows -> {n_c:,} calls + {n_a:,} activities '
                             f'({len(o) / max(n_c, 1):.2f} rows per call); all-null columns dropped: {len(empty)}')
    return calls, acts


def prep_k2(t, name, contract_key=False):
    """K2 card-index (vneshpt_ot = third-party collection orders ИР3/ИР4, vnpt = the bank's own ВнПТ)."""
    n0 = len(t)
    if 'is_actual' in t.columns:
        t = t[t['is_actual'].astype('str').str.strip().str.upper().eq('A')]
    iin = norm_iin(col(t, 'iin_bin'))
    if 'recip_client_iin' in t.columns:
        iin = iin.fillna(norm_iin(t['recip_client_iin']))
    t = t.assign(iin=iin, accept_date=to_dt(t['accept_date']), close_date=to_dt(col(t, 'close_date')),
                 date_change=to_dt(col(t, 'date_change')),
                 amount_sum=pd.to_numeric(col(t, 'amount_sum'), errors='coerce'))
    if contract_key:
        cn = norm_contract(col(t, 'dogovor'))
        if 'doc_num' in t.columns:
            cn = cn.fillna(norm_contract(t['doc_num']))
        t = t.assign(contract_number=cn)
    if 'req_id' in t.columns:                 # SCD2: one (latest) version per demand
        t = t.sort_values('date_change', na_position='first').drop_duplicates('req_id', keep='last')
    t = t.dropna(subset=['accept_date']).reset_index(drop=True)
    DQ_NOTES[name] = f'{n0 - len(t):,} rows dropped (not actual / old SCD versions); open (close_date NaT): {t["close_date"].isna().mean():.0%}'
    return t


EXCL_CATEGORIES = [('bankruptcy', r'банкрот'), ('socially_vulnerable', r'СУСН'), ('uvek_letter_2020', r'УВЕК'),
                   ('simplified', r'упращ|упрощ'), ('bizbirgemiz', r'БизБиргемиз|Бiз|Біз')]
EXCL_ALL = [k for k, _ in EXCL_CATEGORIES] + ['other']


def prep_excl(e):
    if 'is_actual' in e.columns:
        e = e[e['is_actual'].astype('str').str.strip().str.upper().eq('A')]
    title = e['exclude_title'].astype('str').str.strip()
    cat = np.select([title.str.contains(p, case=False, regex=True, na=False).to_numpy() for _, p in EXCL_CATEGORIES],
                    [k for k, _ in EXCL_CATEGORIES], default='other')
    e = e.assign(iin=norm_iin(e['iin_bin']), excl_category=pd.Series(cat, index=e.index, dtype='str'),
                 title_norm=title.str.upper(), is_active_fl=pd.to_numeric(col(e, 'is_active_fl'), errors='coerce'),
                 updated_date=to_dt(col(e, 'updated_date')))
    n0 = len(e)
    e = (e.dropna(subset=['iin']).sort_values('updated_date', na_position='first')
          .drop_duplicates(['iin', 'excl_category', 'title_norm'], keep='last').reset_index(drop=True))
    DQ_NOTES['excl'] = f'{n0 - len(e):,} duplicate rows dropped; categories {e["excl_category"].value_counts().to_dict()}'
    return e


def prep_vneshpt(x):
    x = x.rename(columns={'РЕФЕРЕНС': 'reference', 'ИИН': 'iin_raw', 'Тип бизнеса': 'business_type', 'Причина': 'reason'})
    n0 = len(x)
    x = x.drop_duplicates([c for c in ('reference', 'longname', 'status_code', 'date_change', 'debt') if c in x.columns])
    n_dup = n0 - len(x)
    x = x.assign(iin=norm_iin(col(x, 'iin_raw')), contract_number=norm_contract(col(x, 'ext_dea_code')),
                 created_date=to_dt(col(x, 'created_date')), date_change=to_dt(col(x, 'date_change')),
                 debt=pd.to_numeric(col(x, 'debt'), errors='coerce'))
    x = x.assign(ev_date=coalesce(x['date_change'], x['created_date']),
                 created_date=coalesce(x['created_date'], x['date_change']))
    DQ_NOTES['vneshpt'] = f'exact duplicates dropped FIRST: {n_dup:,} ({n_dup / max(n0, 1):.0%} of rows)'
    return x.dropna(subset=['iin', 'ev_date']).reset_index(drop=True)


def prep_geo(g):
    """App logins -> daily aggregates per client (+ distinct device / IP per day)."""
    lat_min, lat_max, lon_min, lon_max = CONFIG['KZ_BBOX']
    g = g.assign(iin=norm_iin(g['client_iin']),
                 ts=coalesce(to_dt(col(g, 'event_datetime')), to_dt(col(g, 'date_value')))).dropna(subset=['iin', 'ts'])
    lat = pd.to_numeric(col(g, 'latitude'), errors='coerce')
    lon = pd.to_numeric(col(g, 'longitude'), errors='coerce')
    valid = lat.notna() & lon.notna() & ~((lat == 0) & (lon == 0))        # 0/0 = missing geolocation
    inside = lat.between(lat_min, lat_max) & lon.between(lon_min, lon_max)
    act = col(g, 'action').astype('str').str.strip().str.lower()
    g = g.assign(day=g['ts'].dt.normalize(),
                 bio=act.isin(['face_id', 'touch_id']).astype('int32'),
                 night=(g['ts'].dt.hour < 6).astype('int32'),
                 valid=valid.astype('int32'), abroad=(valid & ~inside).astype('int32'),
                 premium=col(g, 'device_name').astype('str').str.contains(r'\b(?:Pro|Max|Ultra)\b', case=False,
                                                                         regex=True, na=False).astype('int32'),
                 app_major=pd.to_numeric(col(g, 'app_version_name').astype('str').str.extract(r'^\s*(\d+)', expand=False),
                                         errors='coerce'))
    daily = (g.groupby(['iin', 'day'], sort=False)
              .agg(n=('ts', 'size'), n_bio=('bio', 'sum'), n_night=('night', 'sum'), n_valid=('valid', 'sum'),
                   n_abroad=('abroad', 'sum'), premium=('premium', 'max'), app_major=('app_major', 'max'))
              .reset_index())
    dev = g[['iin', 'day', 'device_id']].dropna().drop_duplicates() if 'device_id' in g.columns else None
    ip = g[['iin', 'day', 'ip_address']].dropna().drop_duplicates() if 'ip_address' in g.columns else None
    DQ_NOTES['geo'] = (f'{len(g):,} logins -> {len(daily):,} client-days; coordinates 0/0 (missing): '
                       f'{1 - valid.mean():.0%}; abroad logins: {(valid & ~inside).mean():.2%}')
    return daily, dev, ip


DATE_CANDIDATES = ['processcreate', 'date_value', 'date_change', 'lastmodificationdate', 'createdon', 'created_date']
IIN_CANDIDATES = ['iin', 'IIN', 'client_iin', 'iin_bin']
ID_LIKE = re.compile(r'(^|_)(id|iin|bin|inn|rnn|fio|name|mobile|phone|tel|email|ref|orderid|clientid|processid)(_|$)'
                     r'|_id$|^id', re.I)


def detect_col(df, candidates):
    return next((c for c in candidates if c in df.columns), None)


def detect_date_col(df, candidates=DATE_CANDIDATES, min_share=0.8):
    """First candidate column with >= 80% parseable dates."""
    for c in candidates:
        if c in df.columns and to_dt(df[c]).notna().mean() >= min_share:
            return c
    return None


def _looks_like_id(v):
    v = v.dropna()
    if len(v) == 0:
        return False
    integral = np.mean(np.isclose(v.to_numpy() % 1, 0)) > 0.99
    return bool(integral and v.abs().median() >= 1e8 and v.nunique() / len(v) > 0.3)


def prep_dm(d, max_num=40, max_card=30):
    """Approved applications: profile the 159 (mostly text) columns once -> numeric + small categoricals."""
    dcol = detect_date_col(d)
    d = d.assign(process_ref=norm_ref(col(d, 'process_ref')),
                 app_date=to_dt(d[dcol]) if dcol else pd.Series(pd.NaT, index=d.index, dtype='datetime64[us]'))
    base = [c for c in ('request_amount', 'amount_duration', 'response_result') if c in d.columns]
    skip = set(base) | {dcol, 'process_ref', 'app_date'} | set(DATE_CANDIDATES) | set(IIN_CANDIDATES)
    num, cats, stats = {}, [], []
    for c in text_cols(d):
        if c in skip or ID_LIKE.search(c) or re.search(r'close_date|status_name|restructing', c, re.I):
            continue
        s = d[c]
        nn = int(s.notna().sum())
        if nn < 0.2 * len(d):
            continue
        txt = s.astype('str').str.strip()
        v = pd.to_numeric(txt.str.replace(',', '.', regex=False), errors='coerce')
        if v.notna().sum() >= 0.9 * nn:                              # >= 90% numeric-parsable text
            if not _looks_like_id(v):
                num[c] = v.astype('float32')
                stats.append((c, nn, v.nunique()))
            continue
        smp = txt.dropna().head(2000)
        if len(smp) and pd.to_datetime(smp, errors='coerce', format='ISO8601').notna().mean() >= 0.8:
            continue                                                 # date-like text
        if 2 <= s.nunique() <= max_card:
            cats.append(c)
    keep_num = [c for c, _, _ in sorted(stats, key=lambda t: (-t[1], -t[2]))[:max_num]]
    res = pd.concat([d[['process_ref', 'app_date']],
                     pd.DataFrame({c: pd.to_numeric(d[c], errors='coerce') for c in base}, index=d.index).astype('float32'),
                     pd.DataFrame({c: num[c] for c in keep_num}, index=d.index),
                     pd.DataFrame({f'cat_{c}': as_cat(d[c]) for c in cats}, index=d.index)], axis=1)
    res = (res.dropna(subset=['process_ref']).sort_values('app_date', na_position='first')
              .drop_duplicates('process_ref', keep='last').reset_index(drop=True))
    DQ_NOTES['dm'] = (f'date col {dcol!r}; {len(text_cols(d))} text cols -> {len(keep_num)} numeric '
                      f'(>=90% parsable, cap {max_num}) + {len(cats)} categoricals')
    return res


def prep_rej(r):
    icol, dcol = detect_col(r, IIN_CANDIDATES), detect_date_col(r)
    if icol is None or dcol is None:
        raise ValueError(f'dm_rej: iin column {icol!r} / date column {dcol!r} not detected')
    out = pd.DataFrame({'iin': norm_iin(r[icol]), 'app_date': to_dt(r[dcol]),
                        'request_amount': pd.to_numeric(col(r, 'request_amount'), errors='coerce')})
    DQ_NOTES['dm_rej'] = f'iin col {icol!r}, date col {dcol!r}'
    return out.dropna(subset=['iin', 'app_date']).reset_index(drop=True)


# PKB monitoring rule names -> codes (substring match, first match wins; names may be truncated with '...')
PKB_RULE_CODES = [
    ('unsubscribe', r'Отписка'), ('comparison', r'сравнени'), ('ludoman', r'Лудоман'),
    ('stop_credit', r'Стоп.?Кредит'), ('conscript', r'Призывник'), ('bankruptcy', r'Банкрот'),
    ('court', r'Суд'), ('assignment_sold', r'переуступ'), ('closed_repaid', r'закрытие займа'),
    ('collateral_change', r'залог'), ('partial_overdue', r'Частичная просрочка'),
    ('overdue_change', r'сумма просрочки'), ('dpd_change', r'дней просрочки'), ('refinance', r'Рефинанс'),
    ('restructuring', r'Реструктур'), ('prolongation', r'Пролонгац'), ('settled', r'урегулирован'),
    ('credit_app', r'кредитной заявки'), ('report_request', r'запрос\w* на кредитный отчет'),
    ('inquiry', r'искал информацию|Любой поставщик'), ('new_loan', r'нового займа'),
    ('new_contract', r'новый контракт'), ('payment_change', r'ежемесячного платежа'),
    ('principal_change', r'основного долга'),
]
PKB_CODES = [k for k, _ in PKB_RULE_CODES if k != 'unsubscribe'] + ['other']
PKB_KEY_CODES = ['ludoman', 'stop_credit', 'bankruptcy', 'court', 'collateral_change', 'assignment_sold',
                 'partial_overdue', 'overdue_change', 'dpd_change']


def pkb_code(names):
    names = pd.Series(names).astype('str')
    u = pd.Series(names.dropna().unique(), dtype='str')
    code_u = np.select([u.str.contains(p, case=False, regex=True, na=False).to_numpy() for _, p in PKB_RULE_CODES],
                       [k for k, _ in PKB_RULE_CODES], default='other')
    return names.map(dict(zip(u, code_u))).astype('str')


def prep_pkb(p):
    p = p.assign(iin=norm_iin(p['iin']), code=pkb_code(p['rule_name']),
                 first_ts=to_dt(col(p, 'first_ts')), last_ts=to_dt(col(p, 'last_ts')))
    p = p.assign(**{c: pd.to_numeric(col(p, c), errors='coerce').fillna(0) for c in ('cnt_7d', 'cnt_30d', 'cnt_60d', 'cnt_all')})
    snap = to_dt(p['snapshot_date']).max() if 'snapshot_date' in p.columns else p['last_ts'].max()
    unmapped = p.loc[p['code'] == 'other', 'rule_name'].unique()[:5].tolist()
    DQ_NOTES['hes_pkb_agg'] = f'snapshot {snap}; rules mapped to {p["code"].nunique()} codes; unmapped: {unmapped}'
    return p[p['code'] != 'unsubscribe'].dropna(subset=['iin']).reset_index(drop=True), snap

In [ ]:
TBL = {}                       # prepared optional sources (keys normalised, population only)


def _prepare(name, fn):
    raw = load_table(name, columns=TABLE_COLUMNS.get(name),
                     key_filters=[(c, kind, POP_KEYS[kind]) for c, kind in TABLE_KEYS.get(name, [])])
    if raw is None:
        return
    try:
        fn(raw)
    except Exception as e:     # a malformed optional source must not stop the run
        if CONFIG['STRICT_BLOCKS']:
            raise
        print(f'WARN  {name}: preparation failed ({type(e).__name__}: {e}) -> source skipped')


def _put_ocrm(raw):
    TBL['ocrm_calls'], TBL['ocrm_acts'] = prep_ocrm(raw)


def _put_geo(raw):
    TBL['geo_daily'], TBL['geo_dev'], TBL['geo_ip'] = prep_geo(raw)


def _put_pkb(raw):
    TBL['pkb'], TBL['pkb_snapshot'] = prep_pkb(raw)


with timer('optional sources loaded (population keys only) and normalised'):
    _prepare('ver_df_all_2024_2026', lambda d: TBL.update(ver=prep_ver(d)))
    _prepare('bankrupt', lambda d: TBL.update(bankrupt=prep_bankrupt(d)))
    _prepare('ocrm_call', _put_ocrm)
    _prepare('vneshpt_ot', lambda d: TBL.update(vneshpt_ot=prep_k2(d, 'vneshpt_ot')))
    _prepare('vnpt', lambda d: TBL.update(vnpt=prep_k2(d, 'vnpt', contract_key=True)))
    _prepare('excl', lambda d: TBL.update(excl=prep_excl(d)))
    _prepare('vneshpt', lambda d: TBL.update(vneshpt=prep_vneshpt(d)))
    _prepare('geo', _put_geo)
    _prepare('dm', lambda d: TBL.update(dm=prep_dm(d)))
    _prepare('dm_rej', lambda d: TBL.update(dm_rej=prep_rej(d)))
    _prepare('hes_pkb_agg', _put_pkb)


def _rng(s):
    s = pd.Series(s).dropna()
    return (s.quantile(0.01), s.max()) if len(s) else (pd.NaT, pd.NaT)


# Coverage of each source in time: (start = 1% quantile of its event dates, end). Used for the as-of
# masking of blocks and for the MODEL_BLOCKS='auto' rule.
_SRC_DATE = {  # source table -> (prepared table key, date column)
    'ver_df_all_2024_2026': ('ver', 'DATE_VALUE'), 'bankrupt': ('bankrupt', 'event_date'),
    'ocrm_call': ('ocrm_calls', 'call_ts'), 'vneshpt_ot': ('vneshpt_ot', 'accept_date'),
    'vnpt': ('vnpt', 'date_change'), 'vneshpt': ('vneshpt', 'ev_date'), 'geo': ('geo_daily', 'day'),
    'dm': ('dm', 'app_date'), 'dm_rej': ('dm_rej', 'app_date'), 'hes_pkb_agg': ('pkb', 'first_ts'),
}
SRC_RANGE = {'contr': _rng(contracts['open_date']), 'ovrd': _rng(ovrd_clean['act_date'])}
for _src, (_key, _dc) in _SRC_DATE.items():
    if _key in TBL:
        _s = TBL[_key][_dc]
        if _src == 'vnpt' and _s.isna().all():
            _s = TBL[_key]['accept_date']
        SRC_RANGE[_src] = _rng(_s)
if 'excl' in TBL:              # a snapshot list: no event history
    SRC_RANGE['excl'] = (pd.NaT, TBL['excl']['updated_date'].max())
SRC_START = {k: v[0] for k, v in SRC_RANGE.items()}

In [ ]:
# ---- data-quality report: one row per source ------------------------------------------------------
_DQ_SPEC = [  # (source, prepared frame, key column, natural-key columns, date column)
    ('contr', contracts, 'contract_number', ['contract_number'], 'open_date'),
    ('ovrd', ovrd_clean, 'contract_number', ['contract_number', 'rep_month'], 'act_date'),
    ('ver_df_all_2024_2026', TBL.get('ver'), 'iin', ['iin', 'DATE_VALUE'], 'DATE_VALUE'),
    ('bankrupt', TBL.get('bankrupt'), 'iin', ['iin', 'event_date'], 'event_date'),
    ('ocrm_call', TBL.get('ocrm_calls'), 'contract_number', ['callid'], 'call_ts'),
    ('vneshpt_ot', TBL.get('vneshpt_ot'), 'iin', ['iin', 'accept_date', 'amount_sum'], 'accept_date'),
    ('vnpt', TBL.get('vnpt'), 'contract_number', ['contract_number', 'accept_date', 'amount_sum'], 'accept_date'),
    ('excl', TBL.get('excl'), 'iin', ['iin', 'excl_category'], 'updated_date'),
    ('vneshpt', TBL.get('vneshpt'), 'iin', ['reference', 'status_code', 'date_change'], 'ev_date'),
    ('geo', TBL.get('geo_daily'), 'iin', ['iin', 'day'], 'day'),
    ('dm', TBL.get('dm'), 'process_ref', ['process_ref'], 'app_date'),
    ('dm_rej', TBL.get('dm_rej'), 'iin', ['iin', 'app_date'], 'app_date'),
    ('hes_pkb_agg', TBL.get('pkb'), 'iin', ['iin', 'code'], 'last_ts'),
]
_loaded = {r['table']: r for r in LOAD_LOG}
_rows = []
for _name, _df, _key, _nk, _dc in _DQ_SPEC:
    _r = _loaded.get(_name, {})
    _row = dict(table=_name, source=_r.get('source', 'loaded'), rows_raw=_r.get('rows_raw', np.nan),
                rows_population=_r.get('rows_loaded', np.nan), rows_prepared=np.nan, key=_key,
                key_null_pct=np.nan, n_keys=np.nan, dup_natural_key=np.nan, date_min=pd.NaT, date_max=pd.NaT,
                notes=DQ_NOTES.get(_name, ''))
    if _df is not None and _key in _df.columns:
        _nk = [c for c in _nk if c in _df.columns]
        _d = _df[_dc].dropna() if _dc in _df.columns else pd.Series(dtype='datetime64[us]')
        _row.update(rows_prepared=len(_df), key_null_pct=round(100 * _df[_key].isna().mean(), 2),
                    n_keys=_df[_key].nunique(), dup_natural_key=int(_df.duplicated(_nk).sum()) if _nk else np.nan,
                    date_min=_d.min() if len(_d) else pd.NaT, date_max=_d.max() if len(_d) else pd.NaT)
    _rows.append(_row)
DQ_REPORT = pd.DataFrame(_rows).set_index('table')
DQ_REPORT = DQ_REPORT.astype({c: 'Int64' for c in ('rows_raw', 'rows_population', 'rows_prepared', 'n_keys', 'dup_natural_key')})
show(DQ_REPORT, 'Data-quality report (population-filtered extracts)')

## 4. Monthly DPD panel and soft-collection entry episodes
- **Panel.** One row per (contract, `rep_month`): the row with the maximum DPD, with its `act_date`. The implied **start of the overdue** is `start_date = act_date - (dpd - 1)`.
- **Episodes.** Rows are sorted by contract and `start_date`. A row starts a new episode when its `start_date` is more than `EPISODE_GAP_DAYS` after the previous row's. `entry_date` is the episode's first `start_date`, and `obs_date = entry_date + OBS_OFFSET_DAYS`. Features use **only events dated before `obs_date`**.
- **Targets.**
  - `bad = 1` if any row of the contract reaches 90 DPD on a date `start_date + 89` inside `(obs_date, obs_date + 6 months]`.
  - `roll31_60d` is the same test at 31+ DPD within 60 days.
  - `self_cure30` means the episode's maximum DPD is at most 30.
- **Labelled** episodes have a full 6-month window before the data cut (the maximum `act_date`).
- **Current stock.** Contracts whose row in the latest `rep_month` has DPD 1-30 and is still overdue on the cut. They are scored on their **current episode's** entry features, and their rule overlays use the latest information at `score_date = cut + 1 day`.

In [ ]:
_p = ovrd_clean.sort_values(['contract_number', 'rep_month', 'dpd', 'act_date'])
panel = _p.drop_duplicates(['contract_number', 'rep_month'], keep='last')
_n_multi = len(_p) - len(panel)
panel = panel.assign(start_date=(panel['act_date'].dt.normalize()
                                 - pd.to_timedelta(panel['dpd'] - 1, unit='D')).astype('datetime64[us]'))
panel = panel.sort_values(['contract_number', 'start_date', 'act_date']).reset_index(drop=True)
_prev = panel.groupby('contract_number', sort=False)['start_date'].shift()
panel['new_ep'] = _prev.isna() | ((panel['start_date'] - _prev).dt.days > CONFIG['EPISODE_GAP_DAYS'])
panel['episode_id'] = panel['new_ep'].cumsum().astype('int64') - 1

EP = panel.groupby('episode_id').agg(contract_number=('contract_number', 'first'), entry_date=('start_date', 'min'),
                                     max_dpd=('dpd', 'max'), last_act_date=('act_date', 'max'),
                                     n_months=('rep_month', 'size'))
_prev_last = EP.groupby('contract_number')['last_act_date'].shift()
EP['reaged'] = (EP['entry_date'] <= _prev_last).astype('int8')     # started before the previous one ended
EP['obs_date'] = (EP['entry_date'] + pd.Timedelta(days=CONFIG['OBS_OFFSET_DAYS'])).astype('datetime64[us]')
DATA_CUT = panel['act_date'].max().normalize()
SCORE_DATE = DATA_CUT + pd.Timedelta(days=1)
log(f'panel: {len(panel):,} contract-months ({_n_multi} duplicate contract-month rows collapsed to the max DPD); '
    f'{len(EP):,} episodes on {EP["contract_number"].nunique():,} contracts; data cut {DATA_CUT:%Y-%m-%d}')

In [ ]:
def _hits(thr, end_col):
    """Episodes with a contract row reaching `thr` DPD on a date in (obs_date, end]."""
    h = panel.loc[panel['dpd'] >= thr, ['contract_number', 'start_date']]
    h = h.assign(d=h['start_date'] + pd.Timedelta(days=thr - 1)).drop_duplicates(['contract_number', 'd'])
    m = EP[['contract_number', 'obs_date', end_col]].reset_index().merge(h[['contract_number', 'd']], on='contract_number')
    m = m[(m['d'] > m['obs_date']) & (m['d'] <= m[end_col])]
    return EP.index.isin(m['episode_id'])


EP['horizon_end'] = (EP['obs_date'] + pd.DateOffset(months=CONFIG['HORIZON_MONTHS'])).astype('datetime64[us]')
EP['roll_end'] = (EP['obs_date'] + pd.Timedelta(days=CONFIG['ROLL_DAYS'])).astype('datetime64[us]')
EP['bad'] = _hits(CONFIG['BAD_DPD'], 'horizon_end').astype('int8')
EP['roll31_60d'] = _hits(CONFIG['ROLL_DPD'], 'roll_end').astype('int8')
EP['self_cure30'] = (EP['max_dpd'] <= 30).astype('int8')
EP['labeled'] = EP['horizon_end'] <= DATA_CUT
EP['labeled_roll'] = EP['roll_end'] <= DATA_CUT
EP['obs_month'] = EP['obs_date'].dt.to_period('M')

_lab = EP[EP['labeled']]
MONTHLY = EP.groupby('obs_month').agg(entries=('bad', 'size'), labeled=('labeled', 'sum'))
MONTHLY['bad_rate'] = _lab.groupby('obs_month')['bad'].mean()
MONTHLY['self_cure_rate'] = _lab.groupby('obs_month')['self_cure30'].mean()
MONTHLY['roll31_rate'] = EP[EP['labeled_roll']].groupby('obs_month')['roll31_60d'].mean()
show(MONTHLY, 'Entries into delinquency by month (bad / self-cure on labelled episodes only)')
print(f"episodes {len(EP):,} | labelled {len(_lab):,} | bad rate {_lab['bad'].mean():.1%} | "
      f"self-cure {_lab['self_cure30'].mean():.1%} | roll31_60d {EP.loc[EP['labeled_roll'], 'roll31_60d'].mean():.1%} | "
      f"re-aged (partial payment, overlapping) {EP['reaged'].sum():,}")

In [ ]:
_last_m = panel['rep_month'].max()
_cur = panel[(panel['rep_month'] == _last_m) & panel['dpd'].between(CONFIG['SOFT_DPD_MIN'], CONFIG['SOFT_DPD_MAX'])]
_n_soft = len(_cur)
if CONFIG['STOCK_ONLY_AT_CUT']:
    _cur = _cur[_cur['act_date'].dt.normalize() >= DATA_CUT]
STOCK = EP.loc[_cur['episode_id']].assign(dpd_now=_cur['dpd'].to_numpy(), dpd_asof=_cur['act_date'].to_numpy())
print(f'latest rep_month {_last_m:%Y-%m} (cut {DATA_CUT:%Y-%m-%d}): {_n_soft} rows with DPD '
      f"{CONFIG['SOFT_DPD_MIN']}-{CONFIG['SOFT_DPD_MAX']}, {len(STOCK)} still overdue on the cut -> current stock; "
      f'score_date {SCORE_DATE:%Y-%m-%d}')

# observation frames: history (every episode, features at entry) and stock overlay (latest info)
OBS = (EP.reset_index()[['episode_id', 'contract_number', 'obs_date']].rename(columns={'episode_id': 'obs_id'})
         .merge(contracts[['contract_number', 'process_ref', 'iin']], on='contract_number', how='left'))
OBS_NOW = (OBS[OBS['obs_id'].isin(STOCK.index)].assign(obs_date=SCORE_DATE, overlay=True)
           .astype({'obs_date': 'datetime64[us]'}).reset_index(drop=True))

## 5. Feature blocks
Each block is a function `fb_x(obs, ctx) -> DataFrame` indexed by `obs_id`, with a block prefix on every column. `obs` has the columns `[obs_id, contract_number, process_ref, iin, obs_date]`.

Every block follows the same pattern:
1. Filter the source to the population keys.
2. Merge it with `obs` on the key.
3. Keep events dated **strictly before** `obs_date`, using 30/90/180/365-day windows.
4. Aggregate with `groupby(obs_id)`.

There are no row-wise Python loops and no merges without a key filter. Count features are 0 when nothing happened. All features of a source are NaN before that source starts, so a missing history never reads as zero activity.

To add a source: write `fb_new(obs, ctx)`, register it in `BLOCKS` and add its table to `TABLE_COLUMNS` / `TABLE_KEYS`.

| block | source | prefix | content |
|---|---|---|---|
| a contract | contr | `ct_` | amount, term, MOB, remaining term, dealer, instalment proxy |
| b dpd_history | ovrd | `dh_` | prior episodes, max DPD, months in arrears 3/6/12m, FPD |
| c profile | ver_df | `pr_` | latest snapshot before obs (171 numbers, flags, categories), 90-day trends, income proxy |
| d internal_debits | vnpt | `vn_` | the bank's own payment demands: counts, open, days to close |
| e enforcement | vneshpt_ot | `en_` | third-party collection orders ИР3/ИР4 (bailiffs, tax) |
| f ext_debits | vneshpt | `ex_` | direct debits sent to other banks, reasons (e.g. insufficient funds) |
| g calls | ocrm_call | `cl_` | calls / activities: contact rate, PTP, broken PTP, third party, abroad notes |
| h app_geo | geo | `geo_` | Homebank logins, devices, biometrics, night, abroad share |
| i exclusions | excl | `exc_` | active exclusion lists (bankruptcy, СУСН, УВЕК, ...) |
| j bankruptcy | bankrupt | `bk_` | refused out-of-court bankruptcy applications |
| k applications | dm / dm_rej | `app_` / `rej_` | application data of the contract; rejected applications before obs |
| l pkb_monitoring | PKB agg | `pkb_` | bureau events (overlay only: history starts 2026-09-07) |

In [ ]:
def events_before(obs, ev, key, date_col, cols=()):
    """obs x events joined on `key`, keeping events dated strictly before obs_date; adds age_d (days)."""
    cols = [c for c in dict.fromkeys(cols) if c in ev.columns and c not in (key, date_col)]
    e = ev.loc[ev[key].isin(obs[key].dropna().unique()), [key, date_col, *cols]]
    m = obs[['obs_id', key, 'obs_date']].merge(e, on=key, how='inner')
    m = m[m[date_col] < m['obs_date']]
    return m.assign(age_d=(m['obs_date'] - m[date_col]) / pd.Timedelta(days=1))


def n_within(m, days):
    return m.loc[m['age_d'] <= days].groupby('obs_id').size()


def finish(obs, F, prefix, counts=(), src_start=None):
    """Align a block to obs ids, float32, counts -> 0, everything NaN where obs_date <= source start."""
    F = F.reindex(pd.Index(obs['obs_id'], name='obs_id'))
    F = F.astype({c: 'float32' for c in F.columns if pd.api.types.is_numeric_dtype(F[c])})
    for c in counts:
        if c in F.columns:
            F[c] = F[c].fillna(0)
    if src_start is not None and pd.notna(src_start):
        early = (obs['obs_date'] <= src_start).to_numpy()
        if early.any():
            F.loc[early, :] = np.nan
    F.columns = [prefix + c for c in F.columns]
    return F

In [ ]:
def fb_contract(obs, ctx):
    """a) contract terms at obs_date."""
    c = ctx['contr'].set_index('contract_number')
    o = obs.set_index('obs_id')[['contract_number', 'obs_date']].join(
        c[[x for x in ('open_date', 'end_date', 'contract_amount', 'partner_name', 'product_id', 'repayment_kind',
                       'repayment_day') if x in c.columns]], on='contract_number')
    amt = pd.to_numeric(col(o, 'contract_amount'), errors='coerce')
    term = ((o['end_date'] - o['open_date']).dt.days / 30.4375).round()
    F = pd.DataFrame({
        'amount': amt,
        'log_amount': np.log1p(amt.clip(lower=0)),
        'term_months': term,
        'mob_at_obs': (o['obs_date'] - o['open_date']).dt.days / 30.4375,
        'remaining_months': (o['end_date'] - o['obs_date']).dt.days / 30.4375,
        'installment_proxy': amt / term.where(term > 0),
        'repayment_day': pd.to_numeric(col(o, 'repayment_day'), errors='coerce'),
        'repayment_kind': pd.to_numeric(col(o, 'repayment_kind'), errors='coerce'),
        'partner': as_cat(col(o, 'partner_name'), upper=False),
        'product_id': as_cat(col(o, 'product_id')),
    }, index=o.index)
    return finish(obs, F, 'ct_')


def fb_dpd_history(obs, ctx):
    """b) delinquency history of the contract known before obs_date (ovrd rows with act_date < obs)."""
    m = events_before(obs, ctx['panel'], 'contract_number', 'act_date', ['dpd', 'start_date', 'episode_id'])
    m = m.assign(start_age=(m['obs_date'] - m['start_date']) / pd.Timedelta(days=1))
    g = m.groupby('obs_id')
    F = pd.DataFrame({
        'n_prior_episodes': g['episode_id'].nunique(),
        'max_dpd_ever': g['dpd'].max(),
        'months_dpd_3m': n_within(m, 92), 'months_dpd_6m': n_within(m, 183), 'months_dpd_12m': n_within(m, 366),
        'max_dpd_6m': m[m['age_d'] <= 183].groupby('obs_id')['dpd'].max(),
        'max_dpd_12m': m[m['age_d'] <= 366].groupby('obs_id')['dpd'].max(),
        'days_since_prev_entry': g['start_age'].min(),
        'days_since_last_dpd': g['age_d'].min(),
        'prior_ep_mean_max_dpd': m.groupby(['obs_id', 'episode_id'])['dpd'].max().groupby(level='obs_id').mean(),
    })
    F = F.reindex(pd.Index(obs['obs_id'], name='obs_id'))
    for t in (30, 60, 90):
        F[f'ever{t}'] = (F['max_dpd_ever'] >= t).astype('float32')
    open_date = obs['contract_number'].map(ctx['contr'].set_index('contract_number')['open_date'])
    mob = ((obs['obs_date'] - open_date).dt.days / 30.4375).to_numpy()
    F['fpd_flag'] = np.where(np.isnan(mob), np.nan, (mob <= 2).astype('float64'))   # first/second payment default
    return finish(obs, F, 'dh_', counts=['n_prior_episodes', 'max_dpd_ever', 'months_dpd_3m', 'months_dpd_6m',
                                         'months_dpd_12m'])

In [ ]:
PROFILE_FLAGS = ['FLG_DEATH', 'FLG_BLACK_LIST', 'FLG_LOAN_OVERDUE_MORE_1', 'MAU_FIREBASE', 'MAU_AUTORIZATION',
                 'HALYK_EMPLOYEE']
PROFILE_CATS = ['GENDER', 'BRANCH', 'CLIENT_SEGMENT', 'DIGITAL_SEGMENT', 'EDUCATION', 'FAMILY_STATUS',
                'SALARY_CATEGORY_BY_MODEL', 'TRANSACTION_CLIENT_SEGMENT', 'MOBILE_OS', 'CITIZENSHIP']
TREND_VARS = ['DEPO_SUM', 'PKB_SUM_UNPAY_ACT', 'SUM_PKB_OVRD_ACT', 'VOLUME_INBOX_TRANSF_ANOT_BANK',
              'VOLUME_INBOX_TRANSF_KASPI', 'VOLUME_INBOX_TRANSF_HALYK', 'CNT_SESSIONS', 'CRED_REMAIN_LOAN_SUM',
              'PKB_REQ_30D', 'AMOUNT_LOAN_OVERDUE', 'CNT_PAYMENT', 'MONTH_LOAN_PAY_AMOUNT']
INCOME_VARS = ['VOLUME_INBOX_TRANSF_ANOT_BANK', 'VOLUME_INBOX_TRANSF_KASPI', 'VOLUME_INBOX_TRANSF_HALYK',
               'AMOUNT_SOC_PAYOUT', 'AMOUNT_PENSION']
HOME_COUNTRY = {'KAZAKHSTAN', 'КАЗАХСТАН', 'ҚАЗАҚСТАН', 'KZ', 'KAZ'}


def _asof_snapshot(obs, v, cols, shift_days=0):
    """Latest ver_df snapshot with DATE_VALUE < obs_date - shift_days, per IIN (merge_asof, [us] on both sides)."""
    left = obs[['obs_id', 'iin', 'obs_date']].dropna(subset=['iin'])
    left = left.assign(_k=(left['obs_date'] - pd.Timedelta(days=shift_days)).astype('datetime64[us]')).sort_values('_k')
    right = v.loc[v['iin'].isin(left['iin'].unique()), ['iin', 'DATE_VALUE', *cols]].sort_values('DATE_VALUE')
    m = pd.merge_asof(left, right, left_on='_k', right_on='DATE_VALUE', by='iin', direction='backward',
                      allow_exact_matches=False)
    return m.set_index('obs_id')


def norm_branch(s):
    x = as_cat(s).str.replace(r'\s+', ' ', regex=True)
    return x.str.replace('ОБЛАСТНОЙ ФИЛИАЛ', 'ОФ', regex=False).str.replace('ГОРОДСКОЙ ФИЛИАЛ', 'ГФ', regex=False)


def fb_profile(obs, ctx):
    """c) client profile: latest snapshot strictly before obs_date (+ 90-day trends, derived income)."""
    v = ctx['tables'].get('ver')
    if v is None:
        return None
    lag = CONFIG['PROFILE_LAG_DAYS']
    num = [c for c in v.columns if c not in ('iin', 'DATE_VALUE') and pd.api.types.is_numeric_dtype(v[c])]
    flags = [c for c in PROFILE_FLAGS if c in v.columns and not pd.api.types.is_numeric_dtype(v[c])]
    cats = [c for c in PROFILE_CATS if c in v.columns]
    extra = [c for c in ('DOC_EXPIRY_DATE', 'LAST_EVENT_COUNTRY') if c in v.columns]
    m = _asof_snapshot(obs, v, num + flags + cats + extra, lag)
    new = {f'flag_{c}': flag01(m[c]) for c in flags}
    new.update({f'cat_{c}': (norm_branch(m[c]) if c == 'BRANCH' else as_cat(m[c])) for c in cats})
    new['snapshot_age_days'] = (m['obs_date'] - m['DATE_VALUE']).dt.days
    if 'DOC_EXPIRY_DATE' in m.columns:
        new['doc_days_to_expiry'] = (m['DOC_EXPIRY_DATE'] - m['obs_date']).dt.days
    if 'LAST_EVENT_COUNTRY' in m.columns:
        cty = as_cat(m['LAST_EVENT_COUNTRY'])
        new['last_event_abroad'] = pd.Series(np.where(cty.isna(), np.nan, (~cty.isin(HOME_COUNTRY)).astype(float)),
                                             index=m.index)
    inc = [c for c in INCOME_VARS if c in num]
    new['income_proxy'] = m[inc].astype('float64').sum(axis=1, min_count=1) if inc else np.nan
    tv = [c for c in TREND_VARS if c in num]
    if tv:                                  # second as-of merge at obs_date - 90 days -> deltas / ratios
        m90 = _asof_snapshot(obs, v, tv, lag + 90).reindex(m.index)
        for c in tv:
            now, then = m[c].astype('float64'), m90[c].astype('float64')
            new[f'd90_{c}'] = now - then
            new[f'r90_{c}'] = now / then.where(then.abs() > 0)
    F = pd.concat([m[num], pd.DataFrame(new, index=m.index)], axis=1)
    return finish(obs, F, 'pr_')

In [ ]:
def fb_internal_debits(obs, ctx):
    """d) vnpt: the bank's own payment demands (ВнПТ) - payment discipline. Key: dogovor -> contract; demands on
    accounts that are not a portfolio contract fall back to the client's IIN (decided against the portfolio, not
    against the obs set, so a demand is attributed the same way whatever is being scored)."""
    t = ctx['tables'].get('vnpt')
    if t is None:
        return None
    cols = ['accept_date', 'close_date', 'amount_sum']
    in_c = t['contract_number'].isin(ctx['portfolio_contracts'])
    m = pd.concat([events_before(obs, t[in_c], 'contract_number', 'accept_date', cols),
                   events_before(obs, t[~in_c & t['iin'].notna()], 'iin', 'accept_date', cols)], ignore_index=True)
    m = m.assign(is_open=m['close_date'].isna() | (m['close_date'] >= m['obs_date']))
    closed = m[~m['is_open']]
    closed = closed.assign(ttc=(closed['close_date'] - closed['accept_date']) / pd.Timedelta(days=1),
                           same_day=(closed['close_date'].dt.normalize() == closed['accept_date'].dt.normalize()).astype('float32'))
    closed = closed.assign(within3=(closed['ttc'] <= 3).astype('float32'))
    g, gc, op = m.groupby('obs_id'), closed.groupby('obs_id'), m[m['is_open']].groupby('obs_id')
    r90 = m[m['age_d'] <= 90]
    per_day = r90.groupby([r90['obs_id'], r90['accept_date'].dt.normalize()]).size()
    F = pd.DataFrame({
        'n_30d': n_within(m, 30), 'n_90d': n_within(m, 90), 'n_180d': n_within(m, 180),
        'n_open': op.size(), 'open_amount': op['amount_sum'].sum(),
        'amount_90d': r90.groupby('obs_id')['amount_sum'].sum(),
        'ttc_mean': gc['ttc'].mean(), 'ttc_max': gc['ttc'].max(),
        'ttc_last': closed.sort_values('accept_date').groupby('obs_id')['ttc'].last(),
        'share_same_day': gc['same_day'].mean(), 'share_within_3d': gc['within3'].mean(),
        'days_since_last': g['age_d'].min(),
        'multi_same_day_90d': (per_day >= 2).groupby(level=0).sum(),          # partial-payment proxy
    })
    return finish(obs, F, 'vn_', counts=['n_30d', 'n_90d', 'n_180d', 'n_open', 'open_amount', 'amount_90d',
                                         'multi_same_day_90d'], src_start=SRC_START.get('vnpt'))


def fb_enforcement(obs, ctx):
    """e) vneshpt_ot: third-party collection orders ИР3/ИР4 on the client's accounts (enforcement)."""
    t = ctx['tables'].get('vneshpt_ot')
    if t is None:
        return None
    m = events_before(obs, t, 'iin', 'accept_date',
                      ['close_date', 'amount_sum', 'opertype_name', 'descr', 'k2_requirement_group_id'])
    m = m.assign(is_open=m['close_date'].isna() | (m['close_date'] >= m['obs_date']),
                 ir4=as_cat(col(m, 'opertype_name')).eq('ИР4').astype('float32'),
                 ispoln=col(m, 'descr').astype('str').str.contains('исполнительн', case=False, na=False).astype('float32'),
                 grp=col(m, 'k2_requirement_group_id'))
    g, op = m.groupby('obs_id'), m[m['is_open']].groupby('obs_id')
    F = pd.DataFrame({
        'n_ever': g.size(), 'n_365d': n_within(m, 365), 'n_90d': n_within(m, 90),
        'n_open': op.size(), 'open_amount': op['amount_sum'].sum(), 'max_amount': g['amount_sum'].max(),
        'days_since_last': g['age_d'].min(), 'n_groups': g['grp'].nunique(),
        'share_ir4': g['ir4'].mean(), 'share_enforcement_descr': g['ispoln'].mean(),
    })
    return finish(obs, F, 'en_', counts=['n_ever', 'n_365d', 'n_90d', 'n_open', 'open_amount', 'n_groups'],
                  src_start=SRC_START.get('vneshpt_ot'))


def fb_ext_debits(obs, ctx):
    """f) vneshpt: direct-debit demands sent to the client's OTHER banks (status events before obs)."""
    t = ctx['tables'].get('vneshpt')
    if t is None:
        return None
    m = events_before(obs, t, 'iin', 'ev_date', ['reference', 'longname', 'reason', 'created_date', 'debt'])
    m = m.assign(c_age=(m['obs_date'] - m['created_date']) / pd.Timedelta(days=1))
    w90 = m[m['age_d'] <= 90]
    debt_ref = w90.groupby(['obs_id', 'reference'])['debt'].max()
    cols = {
        'n_demands_30d': m[m['c_age'] <= 30].groupby('obs_id')['reference'].nunique(),
        'n_demands_90d': m[m['c_age'] <= 90].groupby('obs_id')['reference'].nunique(),
        'n_refs_ever': m.groupby('obs_id')['reference'].nunique(),
        'n_banks_90d': w90.groupby('obs_id')['longname'].nunique(),
        'n_banks_ever': m.groupby('obs_id')['longname'].nunique(),
        'debt_sum_90d': debt_ref.groupby(level='obs_id').sum(),
        'debt_max_90d': debt_ref.groupby(level='obs_id').max(),
        'days_since_last': m.groupby('obs_id')['age_d'].min(),
    }
    for reason, label in ctx['ext_reasons'].items():
        cols[f'share_{label}_90d'] = (w90['reason'] == reason).astype('float32').groupby(w90['obs_id']).mean()
    return finish(obs, pd.DataFrame(cols), 'ex_',
                  counts=['n_demands_30d', 'n_demands_90d', 'n_refs_ever', 'n_banks_90d', 'n_banks_ever', 'debt_sum_90d'],
                  src_start=SRC_START.get('vneshpt'))

In [ ]:
ABROAD_RE = r'загран|уехал|за границ'


def fb_calls(obs, ctx):
    """g) ocrm_call split into CALLS (dedup callid) and ACTIVITIES (dedup activityid), key num_dbz -> contract."""
    calls, acts = ctx['tables'].get('ocrm_calls'), ctx['tables'].get('ocrm_acts')
    if calls is None and acts is None:
        return None
    cols = {}
    third = abroad = pd.Series(dtype='float64')
    if calls is not None:
        mc = events_before(obs, calls, 'contract_number', 'call_ts',
                           ['callid', 'talktime', 'result_predl', 'tnumb_to', 'tsptp_date', 'call_comment'])
        res = col(mc, 'result_predl').astype('str').str.strip()
        mc = mc.assign(contacted=pd.to_numeric(col(mc, 'talktime'), errors='coerce').gt(0).astype('float32'),
                       noans=res.eq('Нет ответа').astype('float32'),
                       ptp=res.str.upper().eq('PTP').astype('float32'),
                       third=res.str.contains('трет|3 лиц', case=False, na=False),
                       abroad=col(mc, 'call_comment').astype('str').str.contains(ABROAD_RE, case=False, na=False))
        y = mc[mc['age_d'] <= 365]
        gy = y.groupby('obs_id')
        cols.update({
            'n_calls_30d': n_within(mc, 30), 'n_calls_90d': n_within(mc, 90), 'n_calls_365d': gy.size(),
            'n_contacted_365d': gy['contacted'].sum(), 'contact_rate_365d': gy['contacted'].mean(),
            'n_noanswer_365d': gy['noans'].sum(), 'n_ptp_calls_365d': gy['ptp'].sum(),
            'n_numbers_dialled': mc.groupby('obs_id')['tnumb_to'].nunique() if 'tnumb_to' in mc.columns else None,
            'days_since_call': mc.groupby('obs_id')['age_d'].min(),
            'days_since_contact': mc[mc['contacted'] > 0].groupby('obs_id')['age_d'].min(),
        })
        third = y[y['third']].groupby('obs_id').size()
        abroad = mc[mc['abroad']].groupby('obs_id').size()
        if 'tsptp_date' in mc.columns:       # broken promise: still overdue after the promised date
            pt = mc.loc[mc['tsptp_date'].notna() & (mc['tsptp_date'] < mc['obs_date']),
                        ['obs_id', 'contract_number', 'callid', 'tsptp_date', 'obs_date']].drop_duplicates(['obs_id', 'callid'])
            x = pt.merge(ctx['panel'][['contract_number', 'start_date', 'act_date']], on='contract_number')
            x = x[(x['act_date'] < x['obs_date']) & (x['start_date'] <= x['tsptp_date']) & (x['act_date'] > x['tsptp_date'])]
            cols['n_ptp_due'] = pt.groupby('obs_id').size()
            cols['n_ptp_broken'] = x.drop_duplicates(['obs_id', 'callid']).groupby('obs_id').size()
    if acts is not None:
        ma = events_before(obs, acts, 'contract_number', 'act_ts',
                           ['activ_sost', 'activ_result', 'activ_result_dtl', 'ocrm_tsdebt_id'])
        ares = col(ma, 'activ_result').astype('str')
        ma = ma.assign(canceled=col(ma, 'activ_sost').astype('str').str.strip().str.lower().eq('canceled').astype('float32'),
                       ptp=ares.str.contains('Обещание', case=False, na=False).astype('float32'),
                       third=ares.str.contains('3 лиц', case=False, na=False),
                       abroad=col(ma, 'activ_result_dtl').astype('str').str.contains(ABROAD_RE, case=False, na=False))
        ya = ma[ma['age_d'] <= 365]
        ga = ya.groupby('obs_id')
        cols.update({'n_acts_365d': ga.size(), 'share_canceled_365d': ga['canceled'].mean(),
                     'n_ptp_acts_365d': ga['ptp'].sum(),
                     'n_debt_episodes': ma.groupby('obs_id')['ocrm_tsdebt_id'].nunique() if 'ocrm_tsdebt_id' in ma.columns else None})
        third = third.add(ya[ya['third']].groupby('obs_id').size(), fill_value=0)
        abroad = abroad.add(ma[ma['abroad']].groupby('obs_id').size(), fill_value=0)
    cols['n_third_party_365d'] = third
    cols['abroad_kw'] = (abroad > 0).astype('float32')
    F = pd.DataFrame({k: v for k, v in cols.items() if v is not None})
    F = F.reindex(pd.Index(obs['obs_id'], name='obs_id'))
    counts = [c for c in F.columns if c.startswith('n_')] + ['abroad_kw']
    F[[c for c in counts if c in F.columns]] = F[[c for c in counts if c in F.columns]].fillna(0)
    F['n_ptp_365d'] = F.get('n_ptp_calls_365d', 0) + F.get('n_ptp_acts_365d', 0)
    if 'n_ptp_due' in F.columns:
        F['ptp_broken_share'] = F['n_ptp_broken'] / F['n_ptp_due'].where(F['n_ptp_due'] > 0)
    return finish(obs, F, 'cl_', src_start=SRC_START.get('ocrm_call'))


def fb_app_geo(obs, ctx):
    """h) Homebank logins (pre-aggregated per client-day): activity, trend, devices, biometrics, abroad."""
    d = ctx['tables'].get('geo_daily')
    if d is None:
        return None
    m = events_before(obs, d, 'iin', 'day', ['n', 'n_bio', 'n_night', 'n_valid', 'n_abroad', 'premium', 'app_major'])
    w30, w90 = m[m['age_d'] <= 30], m[m['age_d'] <= 90]
    s30 = w30.groupby('obs_id')[['n', 'n_valid', 'n_abroad']].sum()
    s90 = w90.groupby('obs_id')[['n', 'n_bio', 'n_night', 'n_valid', 'n_abroad']].sum()
    cols = {
        'logins_7d': m[m['age_d'] <= 7].groupby('obs_id')['n'].sum(), 'logins_30d': s30['n'], 'logins_90d': s90['n'],
        'logins_31_90d': w90[w90['age_d'] > 30].groupby('obs_id')['n'].sum(),
        'active_days_30d': w30.groupby('obs_id').size(),
        'days_since_login': m.groupby('obs_id')['age_d'].min(),
        'share_bio_90d': s90['n_bio'] / s90['n'], 'share_night_90d': s90['n_night'] / s90['n'],
        'share_valid_coords_90d': s90['n_valid'] / s90['n'],
        'share_abroad_30d': s30['n_abroad'] / s30['n_valid'].where(s30['n_valid'] > 0),
        'share_abroad_90d': s90['n_abroad'] / s90['n_valid'].where(s90['n_valid'] > 0),
        'premium_device_90d': w90.groupby('obs_id')['premium'].max(),
        'app_major_last': m.sort_values('day').groupby('obs_id')['app_major'].last(),
    }
    for name, key in (('n_devices_90d', 'geo_dev'), ('n_ips_90d', 'geo_ip')):
        t = ctx['tables'].get(key)
        if t is not None:
            k = t.columns[-1]
            x = events_before(obs, t, 'iin', 'day', [k])
            cols[name] = x[x['age_d'] <= 90].groupby('obs_id')[k].nunique()
    F = pd.DataFrame(cols).reindex(pd.Index(obs['obs_id'], name='obs_id'))
    counts = ['logins_7d', 'logins_30d', 'logins_90d', 'logins_31_90d', 'active_days_30d', 'n_devices_90d', 'n_ips_90d']
    F[[c for c in counts if c in F.columns]] = F[[c for c in counts if c in F.columns]].fillna(0)
    F['trend_30_vs_prev60'] = (F['logins_30d'] + 1) / (F['logins_31_90d'] / 2 + 1)      # smoothed ratio
    return finish(obs, F, 'geo_', src_start=SRC_START.get('geo'))

In [ ]:
def fb_exclusions(obs, ctx):
    """i) exclusion lists active at obs_date (is_active_fl == 1 and not updated after obs)."""
    e = ctx['tables'].get('excl')
    if e is None:
        return None
    m = obs[['obs_id', 'iin', 'obs_date']].dropna(subset=['iin']).merge(
        e[['iin', 'excl_category', 'is_active_fl', 'updated_date']], on='iin')
    known = m['updated_date'].isna() | (m['updated_date'] < m['obs_date'])
    act = m[known & m['is_active_fl'].eq(1)]
    piv = (act.groupby(['obs_id', 'excl_category']).size().unstack(fill_value=0).clip(upper=1)
              .reindex(columns=EXCL_ALL, fill_value=0).add_prefix('active_'))
    F = pd.concat([piv, m[known].groupby('obs_id').size().clip(upper=1).rename('listed_ever')], axis=1)
    F = F.reindex(pd.Index(obs['obs_id'], name='obs_id')).fillna(0)
    F['active_any'] = F[[f'active_{c}' for c in EXCL_ALL]].max(axis=1)
    F['active_other_list'] = F[[f'active_{c}' for c in EXCL_ALL if c != 'bankruptcy']].max(axis=1)
    return finish(obs, F, 'exc_')


def fb_bankruptcy(obs, ctx):
    """j) refused out-of-court bankruptcy applications before obs_date."""
    b = ctx['tables'].get('bankrupt')
    if b is None:
        return None
    m = events_before(obs, b, 'iin', 'event_date', ['refused'])
    g = m.groupby('obs_id')
    F = pd.DataFrame({'n_apps': g.size(), 'n_refused': g['refused'].sum(), 'days_since_last': g['age_d'].min()})
    F['flag_365d'] = (F['days_since_last'] <= 365).astype('float32')
    return finish(obs, F, 'bk_', counts=['n_apps', 'n_refused', 'flag_365d'], src_start=SRC_START.get('bankrupt'))


def fb_applications(obs, ctx):
    """k1) dm: the approved application of this contract (process_ref), dated before obs_date."""
    a = ctx['tables'].get('dm')
    if a is None:
        return None
    x = obs[['obs_id', 'process_ref', 'obs_date']].dropna(subset=['process_ref']).merge(a, on='process_ref')
    x = x[x['app_date'].isna() | (x['app_date'] < x['obs_date'])]
    x = x.sort_values('app_date', na_position='first').drop_duplicates('obs_id', keep='last').set_index('obs_id')
    feats = [c for c in a.columns if c not in ('process_ref', 'app_date')]
    F = x[feats].assign(days_app_to_obs=(x['obs_date'] - x['app_date']).dt.days)
    return finish(obs, F, 'app_')


def fb_rejections(obs, ctx):
    """k2) dm_rej: rejected applications of the client before obs_date (credit hunger)."""
    r = ctx['tables'].get('dm_rej')
    if r is None:
        return None
    m = events_before(obs, r, 'iin', 'app_date', ['request_amount'])
    F = pd.DataFrame({'n_30d': n_within(m, 30), 'n_90d': n_within(m, 90), 'n_180d': n_within(m, 180),
                      'n_ever': m.groupby('obs_id').size(), 'days_since_last': m.groupby('obs_id')['age_d'].min(),
                      'sum_requested_180d': m[m['age_d'] <= 180].groupby('obs_id')['request_amount'].sum()})
    return finish(obs, F, 'rej_', counts=['n_30d', 'n_90d', 'n_180d', 'n_ever', 'sum_requested_180d'],
                  src_start=SRC_START.get('dm_rej'))


def fb_pkb_monitoring(obs, ctx):
    """l) PKB monitoring aggregate (long format from the SQL). Only for obs on/after the aggregate's snapshot or
    for the stock overlay; NaN otherwise (history starts 2026-09-07 -> overlay / rules only)."""
    p = ctx['tables'].get('pkb')
    if p is None:
        return None
    avail = (obs['obs_date'] >= ctx['pkb_available_from']).to_numpy() | col(obs, 'overlay', 0).astype(bool).to_numpy()
    agg = p.groupby(['iin', 'code']).agg(cnt_7d=('cnt_7d', 'sum'), cnt_30d=('cnt_30d', 'sum'), cnt_60d=('cnt_60d', 'sum'),
                                         cnt_all=('cnt_all', 'sum'), last_ts=('last_ts', 'max'))
    wide = {}
    for w in ('7d', '30d', '60d', 'all'):
        u = agg[f'cnt_{w}'].unstack('code').reindex(columns=PKB_CODES)
        wide.update({f'{c}_{w}': u[c] for c in PKB_CODES})
    W = pd.DataFrame(wide)
    last = agg['last_ts'].unstack('code').reindex(columns=PKB_CODES)
    o = obs.loc[avail, ['obs_id', 'iin', 'obs_date']]
    counts = o[['iin']].join(W, on='iin').drop(columns='iin').fillna(0)
    days = pd.DataFrame({f'days_since_{c}': (o['obs_date'] - o['iin'].map(last[c])).dt.days for c in PKB_KEY_CODES},
                        index=o.index)
    F = pd.concat([counts, days], axis=1).set_index(o['obs_id'])
    return finish(obs, F, 'pkb_')

In [ ]:
BLOCKS = [  # name, function, source table, role: 'model' = always, 'auto' = by source start, 'overlay' = rules only
    dict(name='contract', fn=fb_contract, source='contr', role='model'),
    dict(name='dpd_history', fn=fb_dpd_history, source='ovrd', role='model'),
    dict(name='profile', fn=fb_profile, source='ver_df_all_2024_2026', role='auto'),
    dict(name='internal_debits', fn=fb_internal_debits, source='vnpt', role='auto'),
    dict(name='enforcement', fn=fb_enforcement, source='vneshpt_ot', role='auto'),
    dict(name='ext_debits', fn=fb_ext_debits, source='vneshpt', role='auto'),
    dict(name='calls', fn=fb_calls, source='ocrm_call', role='auto'),
    dict(name='app_geo', fn=fb_app_geo, source='geo', role='auto'),
    dict(name='exclusions', fn=fb_exclusions, source='excl', role='overlay'),     # snapshot list, no history
    dict(name='bankruptcy', fn=fb_bankruptcy, source='bankrupt', role='auto'),
    dict(name='applications', fn=fb_applications, source='dm', role='auto'),
    dict(name='rejections', fn=fb_rejections, source='dm_rej', role='auto'),
    dict(name='pkb_monitoring', fn=fb_pkb_monitoring, source='hes_pkb_agg', role='overlay'),
]

# context shared by the blocks
_ext_reasons = {}
if 'vneshpt' in TBL and 'reason' in TBL['vneshpt'].columns:
    for _i, _r in enumerate(TBL['vneshpt']['reason'].value_counts().head(5).index):
        _mm = re.match(r'\s*(\d+)', str(_r))
        _lbl = f'reason_{_mm.group(1)}' if _mm else f'reason_k{_i}'
        _ext_reasons[_r] = _lbl if _lbl not in _ext_reasons.values() else f'{_lbl}_{_i}'
CTX = dict(contr=contracts, panel=panel, tables=TBL, ext_reasons=_ext_reasons,
           portfolio_contracts=set(contracts['contract_number']),
           pkb_available_from=TBL.get('pkb_snapshot', pd.NaT))


def run_blocks(obs):
    """Run every registered block on an observation frame -> ({block: DataFrame}, log table)."""
    out, rows = {}, []
    for b in BLOCKS:
        t = time.perf_counter()
        try:
            F = b['fn'](obs, CTX) if len(obs) else None
            status = 'ok' if F is not None else ('source missing' if len(obs) else 'no observations')
        except Exception as e:
            if CONFIG['STRICT_BLOCKS'] or b['source'] in REQUIRED_TABLES:
                raise
            F, status = None, f'FAILED: {type(e).__name__}: {e}'
            print(f"WARN  block {b['name']} failed -> skipped\n{traceback.format_exc(limit=3)}")
        if F is not None:
            out[b['name']] = F
        start, end = SRC_RANGE.get(b['source'], (pd.NaT, pd.NaT))
        rows.append(dict(block=b['name'], source=b['source'], role=b['role'], status=status,
                         n_features=0 if F is None else F.shape[1],
                         coverage=np.nan if F is None or F.shape[1] == 0 else round(float(F.notna().any(axis=1).mean()), 3),
                         src_start=start, src_end=end, seconds=round(time.perf_counter() - t, 2)))
    return out, pd.DataFrame(rows).set_index('block')


with timer('feature blocks at entry (every episode)'):
    XB, BLOCK_LOG = run_blocks(OBS)
with timer('feature blocks at score date (current stock, latest info)'):
    XN, _ = run_blocks(OBS_NOW)
show(BLOCK_LOG, 'Feature blocks as of obs_date (coverage = share of episodes with any non-null feature)')

### 5.1 As-of self-test (leakage check)
This test rebuilds the data "as the DWH looked at the start of day T": every event dated on or after T is removed, and later closing dates become unknown. It then recomputes every block for the episodes with `obs_date = T` and requires **identical** features. Any difference means a block reads information from after `obs_date`.

Two blocks are not testable this way. `excl` is a snapshot list with no history, so it is rules-only. PKB is a pre-aggregated snapshot used as an overlay only. A third limitation is conservative: `ovrd` keeps one monthly maximum, so a month whose maximum is reached after T is excluded entirely, although part of it was known at T. That loses some information but never leaks.

In [ ]:
def truncate_tables(T, ob):
    """The prepared sources as they looked at the start of day T (events >= T removed, later closes unknown),
    restricted first to the keys of the tested episodes (cheap at full scale)."""
    cut = {'ver': 'DATE_VALUE', 'vneshpt': 'ev_date', 'ocrm_calls': 'call_ts', 'ocrm_acts': 'act_ts',
           'geo_daily': 'day', 'geo_dev': 'day', 'geo_ip': 'day', 'dm_rej': 'app_date', 'bankrupt': 'event_date'}
    keys = {'iin': set(ob['iin'].dropna()), 'contract_number': set(ob['contract_number']),
            'process_ref': set(ob['process_ref'].dropna())}
    out = {}
    for k, v in TBL.items():
        if isinstance(v, pd.DataFrame) and (k in cut or k in ('vnpt', 'vneshpt_ot', 'dm')):
            m = np.zeros(len(v), dtype=bool)
            for kc, ks in keys.items():
                if kc in v.columns:
                    m |= v[kc].isin(ks).to_numpy()
            v = v[m]
        if k in cut:
            out[k] = v[v[cut[k]] < T]
        elif k in ('vnpt', 'vneshpt_ot'):
            x = v[v['accept_date'] < T]
            out[k] = x.assign(close_date=x['close_date'].where(x['close_date'] < T))
        elif k == 'dm':
            out[k] = v[v['app_date'].isna() | (v['app_date'] < T)]
        else:                                     # excl (snapshot list) / pkb (overlay only): not testable
            out[k] = v
    return out


def _same(a, b):
    if pd.api.types.is_numeric_dtype(a) and pd.api.types.is_numeric_dtype(b):
        return np.isclose(a.to_numpy(dtype='float64'), b.to_numpy(dtype='float64'), equal_nan=True, rtol=1e-6)
    return ((a.astype('str') == b.astype('str')) | (a.isna() & b.isna())).to_numpy()


_dates = EP.loc[EP['labeled'], 'obs_date'].value_counts().head(3).index
_rows = []
for _T in _dates:
    _ob = OBS[OBS['obs_date'] == _T]
    _ctx = dict(CTX, tables=truncate_tables(_T, _ob), panel=panel[panel['act_date'] < _T])
    for _b in BLOCKS:
        if _b['name'] in ('exclusions', 'pkb_monitoring') or _b['name'] not in XB:
            continue
        _A = XB[_b['name']].loc[_ob['obs_id']]
        _B = _b['fn'](_ob, _ctx).reindex(index=_A.index, columns=_A.columns)
        _bad = [c for c in _A.columns if not _same(_A[c], _B[c]).all()]
        _rows.append(dict(obs_date=_T, block=_b['name'], n_obs=len(_ob), n_features=_A.shape[1],
                          n_mismatch=len(_bad), mismatching=', '.join(_bad[:5])))
ASOF_TEST = pd.DataFrame(_rows)
ASOF_OK = bool(len(ASOF_TEST)) and int(ASOF_TEST['n_mismatch'].sum()) == 0
show(ASOF_TEST.groupby('block').agg(n_obs=('n_obs', 'sum'), n_features=('n_features', 'max'),
                                    n_mismatch=('n_mismatch', 'sum'), mismatching=('mismatching', 'max')),
     f"As-of self-test on {len(_dates)} entry dates: {'PASS - no look-ahead' if ASOF_OK else 'FAIL - see mismatching features'}")
if not ASOF_OK:
    print('WARNING: a block uses information from after obs_date - fix it before trusting the model.')

## 6. Feature matrix, time split and feature selection
- **Assembly.** All blocks plus cross-block derived features:
  - `capacity_ratio` = income proxy / max(`MONTH_LOAN_PAY_AMOUNT`, instalment proxy)
  - `depo_cover` = `DEPO_SUM` / instalment
  - `debitable_funds` = max(0, min(50% of deposits, deposits - 2 x the subsistence minimum, ПМ)). This is the legal debit limit from 21.10.2026.
- **Split.** OOT = the last `OOT_MONTHS` labelled obs months. The rest is split into train and valid, stratified by `bad` and **grouped by contract**, so episodes of one contract never sit on both sides.
- **`MODEL_BLOCKS='auto'`.** A block feeds the model only if its source starts on or before the **10th percentile of the train `obs_date`**. Late sources (geo since 2025-09, vnpt since 2026-01, vneshpt since 2026-03, PKB since 2026-09) would otherwise teach the model "source present = recent period". They stay in the overlay rules.
- **Selection.** Features are dropped if they are:
  - more than 98% null or constant on train,
  - unstable, with PSI(train -> OOT) above `PSI_DROP_THRESHOLD`,
  - leakage (`close_date`, `status_name`, `restructing_fl`) or listed in `MODEL_EXCLUDE`.

In [ ]:
DERIVED = ['capacity_ratio', 'depo_cover', 'debitable_funds', 'debitable_cover']
_C = CONFIG['COMPLIANCE']


def assemble(blocks, obs):
    idx = pd.Index(obs['obs_id'], name='obs_id')
    parts = [F for F in blocks.values() if F is not None and F.shape[1]]
    X = pd.concat(parts, axis=1).reindex(idx) if parts else pd.DataFrame(index=idx)
    inst = col(X, 'ct_installment_proxy').astype('float64')
    pay = pd.concat([inst, col(X, 'pr_MONTH_LOAN_PAY_AMOUNT').astype('float64')], axis=1).max(axis=1)
    depo = col(X, 'pr_DEPO_SUM').astype('float64')
    d = {'capacity_ratio': col(X, 'pr_income_proxy').astype('float64') / pay.where(pay > 0),
         'depo_cover': depo / inst.where(inst > 0),
         'debitable_funds': pd.concat([_C['DEBIT_MAX_SHARE'] * depo, depo - _C['DEBIT_PROTECTED']], axis=1)
                              .min(axis=1, skipna=False).clip(lower=0)}
    d['debitable_cover'] = d['debitable_funds'] / inst.where(inst > 0)
    return pd.concat([X, pd.DataFrame(d, index=X.index).astype('float32')], axis=1)


X = assemble(XB, OBS)
XNOW = assemble(XN, OBS_NOW)
FEATURE_BLOCK = {c: b for b, F in XB.items() for c in F.columns} | {c: 'derived' for c in DERIVED}

# ---- time split ---------------------------------------------------------------------------------
_lab_months = sorted(EP.loc[EP['labeled'], 'obs_month'].unique())
OOT_MONTHS = set(_lab_months[-CONFIG['OOT_MONTHS']:])
_is_oot = EP['labeled'] & EP['obs_month'].isin(OOT_MONTHS)
_dev = EP.index[EP['labeled'] & ~_is_oot]
_sgk = StratifiedGroupKFold(n_splits=max(2, round(1 / CONFIG['VALID_SHARE'])), shuffle=True,
                            random_state=CONFIG['RANDOM_STATE'])
_tr, _va = next(_sgk.split(np.zeros(len(_dev)), EP.loc[_dev, 'bad'], groups=EP.loc[_dev, 'contract_number']))
_sample = pd.Series('unlabeled', index=EP.index, dtype='str')
_sample[_dev[_tr]] = 'train'
_sample[_dev[_va]] = 'valid'
_sample[_is_oot] = 'oot'
EP['sample'] = _sample
IDS = {s: EP.index[EP['sample'] == s] for s in ('train', 'valid', 'oot')}
TRAIN_P10 = EP.loc[IDS['train'], 'obs_date'].quantile(0.10)

_split = EP.groupby('sample').agg(n=('bad', 'size'), bad_rate=('bad', 'mean'),
                                  first_month=('obs_month', 'min'), last_month=('obs_month', 'max'))
show(_split.reindex(['train', 'valid', 'oot', 'unlabeled']), 'Time split (valid is grouped by contract)')

In [ ]:
def block_used(b):
    if isinstance(CONFIG['MODEL_BLOCKS'], (list, tuple, set)):
        return b['name'] in CONFIG['MODEL_BLOCKS'], 'listed in MODEL_BLOCKS'
    if b['role'] == 'model':
        return True, 'core block'
    if b['role'] == 'overlay':
        return False, 'overlay only (snapshot list / short history)'
    s = SRC_START.get(b['source'])
    if s is None or pd.isna(s):
        return False, 'no source dates'
    ok = s <= TRAIN_P10
    return ok, f"source starts {s:%Y-%m-%d} {'<=' if ok else '>'} train P10 obs_date {TRAIN_P10:%Y-%m-%d}"


_rows = []
for _b in BLOCKS:
    _F = XB.get(_b['name'])
    _used, _why = block_used(_b) if _F is not None else (False, 'source missing / block skipped')
    _cov = lambda F, ids: round(float(F.loc[ids].notna().any(axis=1).mean()), 3) if F is not None and F.shape[1] and len(ids) else np.nan
    _rows.append(dict(block=_b['name'], source=_b['source'], src_start=SRC_START.get(_b['source'], pd.NaT),
                      n_features=0 if _F is None else _F.shape[1], coverage_train=_cov(_F, IDS['train']),
                      coverage_oot=_cov(_F, IDS['oot']),
                      coverage_stock_now=_cov(XN.get(_b['name']), XN[_b['name']].index) if _b['name'] in XN else np.nan,
                      used_in_model=_used, reason=_why))
BLOCK_COVERAGE = pd.DataFrame(_rows).set_index('block')
MODEL_BLOCKS_USED = BLOCK_COVERAGE.index[BLOCK_COVERAGE['used_in_model']].tolist()
show(BLOCK_COVERAGE, f"Block coverage and model usage (MODEL_BLOCKS={CONFIG['MODEL_BLOCKS']!r})")

In [ ]:
LEAK_RE = re.compile('|'.join(LEAKAGE_COLS) + r'|^(bad|self_cure30|roll31_60d|labeled|max_dpd)$', re.I)
_cand = [c for b in MODEL_BLOCKS_USED for c in XB[b].columns]
if 'profile' in MODEL_BLOCKS_USED:
    _cand += DERIVED
_drop = [dict(feature=c, reason='excluded (CONFIG / leakage guard)', value=np.nan)
         for c in _cand if c in CONFIG['MODEL_EXCLUDE'] or LEAK_RE.search(c)]
_cand = [c for c in _cand if c not in CONFIG['MODEL_EXCLUDE'] and not LEAK_RE.search(c)]

_Xtr = X.loc[IDS['train'], _cand]
_null = _Xtr.isna().mean()
_nun = _Xtr.nunique(dropna=True)
_drop += [dict(feature=c, reason=f"null share > {CONFIG['MAX_NULL_SHARE']}", value=_null[c])
          for c in _cand if _null[c] > CONFIG['MAX_NULL_SHARE']]
_drop += [dict(feature=c, reason='constant on train', value=float(_nun[c]))
          for c in _cand if _nun[c] <= 1 and _null[c] <= CONFIG['MAX_NULL_SHARE']]
_keep = [c for c in _cand if _null[c] <= CONFIG['MAX_NULL_SHARE'] and _nun[c] > 1]
PSI_FEATURES = pd.Series({c: psi(X.loc[IDS['train'], c], X.loc[IDS['oot'], c]) for c in _keep}, dtype='float64')
_unstable = PSI_FEATURES[PSI_FEATURES > CONFIG['PSI_DROP_THRESHOLD']].sort_values(ascending=False)
_drop += [dict(feature=c, reason=f"PSI train->OOT > {CONFIG['PSI_DROP_THRESHOLD']}", value=v) for c, v in _unstable.items()]
FEATURES = [c for c in _keep if c not in _unstable.index]
DROPPED = pd.DataFrame(_drop, columns=['feature', 'reason', 'value'])
DROPPED['block'] = DROPPED['feature'].map(FEATURE_BLOCK)
CAT_FEATURES = [c for c in FEATURES if not pd.api.types.is_numeric_dtype(X[c])]
NUM_FEATURES = [c for c in FEATURES if c not in CAT_FEATURES]

# categoricals: train top-N levels + 'OTHER' (fixed categories -> identical encoding everywhere)
CAT_LEVELS = {}
for _c in CAT_FEATURES:
    _vc = X.loc[IDS['train'], _c].value_counts()
    CAT_LEVELS[_c] = [v for v in _vc.index[:CONFIG['TOP_N_CATEGORIES']] if v != 'OTHER'] + ['OTHER']


def encode(Xs):
    """Model matrix: numeric float32, categoricals as pandas category with the train levels."""
    out = {}
    for c in FEATURES:
        if c in CAT_LEVELS:
            v = Xs[c].astype('str')
            v = v.where(v.isna() | v.isin(CAT_LEVELS[c]), 'OTHER')
            out[c] = pd.Categorical(v, categories=CAT_LEVELS[c])
        else:
            out[c] = Xs[c].astype('float32')
    return pd.DataFrame(out, index=Xs.index)


print(f'candidates {len(_cand) + int((DROPPED["reason"] == "excluded (CONFIG / leakage guard)").sum())} -> model features '
      f'{len(FEATURES)} ({len(NUM_FEATURES)} numeric, {len(CAT_FEATURES)} categorical); dropped: '
      + ', '.join(f'{k}: {v}' for k, v in DROPPED['reason'].value_counts().items()))
show(DROPPED[DROPPED['reason'].str.startswith('PSI')].head(20), 'Unstable features dropped (PSI train -> OOT)')

## 7. PD model: LightGBM, with a logistic-regression benchmark
- **LightGBM**: learning rate 0.03, 31 leaves, `min_child_samples = max(50, 0.5% of train)`, feature and bagging fraction 0.8, L2 = 5. It stops early after 100 rounds without a valid-AUC gain, with at most 3000 rounds. Categoricals are native pandas `category`. If LightGBM is unavailable, sklearn `HistGradientBoostingClassifier` is used instead.
- **Benchmark**: a logistic regression on the top-25 numeric features by gain, with 1/99% clipping (train bounds), median imputation and standardisation.
- **Out-of-fold PD for train episodes.** In-sample PDs of a boosted model are over-fitted. For every train episode, `pd` is therefore replaced by a **5-fold out-of-fold** prediction (folds grouped by contract, same settings and number of rounds). The grade cutpoints, the train grade table, the segment backtest and the monitoring baseline are then all out-of-sample. The final model, trained on all train episodes, scores valid, OOT and the stock.
- **Reported**: AUC, Gini and KS on train (in-sample and out-of-fold) / valid / OOT, the score PSI from train to OOT, and gain importance. Exact TreeSHAP contributions come from LightGBM's `pred_contrib`, so no extra package is needed.

In [ ]:
XE = encode(X)                          # every episode (history + stock at entry)
Y = EP['bad']
Xtr, ytr = XE.loc[IDS['train']], Y.loc[IDS['train']]
Xva, yva = XE.loc[IDS['valid']], Y.loc[IDS['valid']]
USE_LGB = bool(CONFIG['USE_LIGHTGBM'] and HAS_LGB)

with timer('PD model trained'):
    if USE_LGB:
        MODEL = lgb.LGBMClassifier(objective='binary', metric='auc', learning_rate=0.03, num_leaves=31,
                                   min_child_samples=max(50, int(0.005 * len(Xtr))), colsample_bytree=0.8,
                                   subsample=0.8, subsample_freq=1, reg_lambda=5.0, n_estimators=3000,
                                   random_state=CONFIG['RANDOM_STATE'], n_jobs=-1, verbose=-1)
        import inspect
        _eval = (dict(eval_X=(Xva,), eval_y=(yva,)) if 'eval_X' in inspect.signature(MODEL.fit).parameters
                 else dict(eval_set=[(Xva, yva)]))              # lightgbm >= 4.7 renamed eval_set
        MODEL.fit(Xtr, ytr, callbacks=[lgb.early_stopping(100, verbose=False)], **_eval)
        N_ROUNDS = int(MODEL.best_iteration_ or MODEL.n_estimators)
        MODEL_KIND = f'LightGBM ({N_ROUNDS} trees)'
        _fixed = dict(n_estimators=N_ROUNDS)
    else:
        MODEL = HistGradientBoostingClassifier(learning_rate=0.05, max_iter=1000, max_leaf_nodes=31,
                                               min_samples_leaf=max(50, int(0.005 * len(Xtr))), l2_regularization=5.0,
                                               categorical_features='from_dtype', early_stopping=True,
                                               validation_fraction=0.2, n_iter_no_change=50,
                                               random_state=CONFIG['RANDOM_STATE'])
        MODEL.fit(Xtr, ytr)
        N_ROUNDS = int(MODEL.n_iter_)
        MODEL_KIND = f'HistGradientBoosting ({N_ROUNDS} iterations)'
        _fixed = dict(max_iter=N_ROUNDS, early_stopping=False)


def predict_pd(Xe):
    return MODEL.predict_proba(Xe)[:, 1] if len(Xe) else np.array([])


# PD of every episode from the final model; for TRAIN episodes it is replaced by an out-of-fold PD
# (5 folds grouped by contract, same settings and number of rounds). In-sample boosted PDs are over-fitted,
# so grade cutpoints, train grade tables, the backtest and the monitoring baseline would otherwise be biased.
PD_IN_SAMPLE = pd.Series(predict_pd(XE), index=EP.index)
EP['pd'] = PD_IN_SAMPLE
_oof = pd.Series(np.nan, index=IDS['train'], dtype='float64')
_folds = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=CONFIG['RANDOM_STATE'])
with timer('out-of-fold PD for train episodes (5 folds)'):
    for _a, _b in _folds.split(np.zeros(len(Xtr)), ytr, groups=EP.loc[IDS['train'], 'contract_number']):
        _m = clone(MODEL).set_params(**_fixed).fit(Xtr.iloc[_a], ytr.iloc[_a])
        _oof.iloc[_b] = _m.predict_proba(Xtr.iloc[_b])[:, 1]
EP.loc[IDS['train'], 'pd'] = _oof
log(f'{MODEL_KIND}: PD scored for {len(EP):,} episodes (train episodes out-of-fold)')

In [ ]:
if USE_LGB:
    IMP = pd.Series(MODEL.booster_.feature_importance('gain'), index=FEATURES, dtype='float64')
else:
    from sklearn.inspection import permutation_importance
    _pi = permutation_importance(MODEL, Xva, yva, scoring='roc_auc', n_repeats=3, random_state=CONFIG['RANDOM_STATE'])
    IMP = pd.Series(_pi.importances_mean, index=FEATURES, dtype='float64').clip(lower=0)
IMP = IMP.sort_values(ascending=False)

LR_FEATURES = [c for c in IMP.index if c in NUM_FEATURES and IMP[c] > 0][:CONFIG['LR_TOP_N']] or NUM_FEATURES[:CONFIG['LR_TOP_N']]
LR, LR_CLIP = None, None
if LR_FEATURES:                 # benchmark: 1/99% clip (train bounds) -> median impute -> standardise -> logit
    LR_CLIP = (Xtr[LR_FEATURES].astype('float64').quantile(0.01), Xtr[LR_FEATURES].astype('float64').quantile(0.99))

    def lr_input(Xe):
        return Xe[LR_FEATURES].astype('float64').clip(lower=LR_CLIP[0], upper=LR_CLIP[1], axis=1)

    LR = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression(max_iter=2000))
    LR.fit(lr_input(Xtr), ytr)
    EP['pd_lr'] = LR.predict_proba(lr_input(XE))[:, 1]

MAIN = MODEL_KIND.split(' ')[0]
_rows = [dict(model=MAIN, sample='train_in_sample', **perf(Y.loc[IDS['train']], PD_IN_SAMPLE.loc[IDS['train']]))]
for _name, _col in ((MAIN, 'pd'), ('LogReg', 'pd_lr')):
    if _col not in EP.columns:
        continue
    for _s in ('train', 'valid', 'oot'):
        _lbl = 'train_oof' if (_s == 'train' and _name == MAIN) else _s
        _rows.append(dict(model=_name, sample=_lbl, **perf(Y.loc[IDS[_s]], EP.loc[IDS[_s], _col])))
METRICS = pd.DataFrame(_rows)
GINI = METRICS.set_index(['model', 'sample'])['gini']
SCORE_PSI = psi(EP.loc[IDS['train'], 'pd'], EP.loc[IDS['oot'], 'pd'])
show(METRICS.set_index(['model', 'sample']), 'Discrimination by sample (train_oof = honest train performance)')
print(f"OOT Gini: {MODEL_KIND} {GINI.get((MAIN, 'oot'), np.nan):.3f} | LogReg {GINI.get(('LogReg', 'oot'), np.nan):.3f} | "
      f"score PSI train->OOT {SCORE_PSI:.3f} ({'stable' if SCORE_PSI < 0.1 else 'investigate' if SCORE_PSI < 0.25 else 'UNSTABLE'})")

In [ ]:
IMP_TABLE = (IMP / IMP.sum()).rename('gain_share').to_frame()
IMP_TABLE['block'] = IMP_TABLE.index.map(FEATURE_BLOCK)
show(IMP_TABLE.head(30), 'Top-30 features by gain share')

SHAP_OK = USE_LGB
if SHAP_OK and len(IDS['oot']):
    _contrib = MODEL.predict(XE.loc[IDS['oot']], pred_contrib=True)[:, :-1]
    SHAP_OOT = pd.Series(np.abs(_contrib).mean(axis=0), index=FEATURES).sort_values(ascending=False)
    show(SHAP_OOT.head(15).rename('mean |SHAP| (log-odds), OOT').to_frame(), 'TreeSHAP (LightGBM pred_contrib) on OOT')


def top_drivers(Xe, k=3):
    """Per row: the k features pushing PD up the most (TreeSHAP, log-odds)."""
    if not SHAP_OK or not len(Xe):
        return pd.Series('', index=Xe.index, dtype='str')
    cb = MODEL.predict(Xe, pred_contrib=True)[:, :-1]
    names = np.asarray(FEATURES)
    order = np.argsort(-cb, axis=1)[:, :k]
    return pd.Series(['; '.join(f'{names[j]} (+{cb[i, j]:.2f})' for j in order[i] if cb[i, j] > 0)
                      for i in range(len(cb))], index=Xe.index, dtype='str')


MODEL_DIR = OUT / 'model'
if USE_LGB:
    MODEL.booster_.save_model(str(MODEL_DIR / 'lgbm_model.txt'))
_ = (MODEL_DIR / 'features.json').write_text(json.dumps({'features': FEATURES, 'categorical_levels': CAT_LEVELS,
                                                      'lr_features': LR_FEATURES}, ensure_ascii=False, indent=1),
                                          encoding='utf-8')
_ = (MODEL_DIR / 'config.json').write_text(json.dumps(CONFIG, ensure_ascii=False, indent=1, default=str), encoding='utf-8')

## 8. Risk grades G1-G5
The PD cutpoints are the quantiles of the **train** PD at `GRADE_QUANTILES` (40/65/85/95%). The train PD here is the out-of-fold PD, so the cutpoints are not stretched by over-fitting. The cutpoints are then **fixed**: OOT and the stock get the same ones, so grade shares may drift while the meaning of a grade does not.

Each sample shows n, average PD, the actual bad rate and the self-cure rate per grade, followed by a monotonicity check. An inversion between neighbouring grades is tested with a one-sided two-proportion z-test. Small G4/G5 samples can invert by chance; only a *significant* inversion means the ranking is broken.

In [ ]:
GRADES = ['G1', 'G2', 'G3', 'G4', 'G5']
CUTS = np.quantile(EP.loc[IDS['train'], 'pd'], CONFIG['GRADE_QUANTILES'])
if len(np.unique(CUTS)) < len(CUTS):
    print('WARN: tied PD cutpoints (few distinct scores) -> some grades will be empty')


def to_grade(p):
    idx = np.searchsorted(CUTS, np.asarray(p, dtype='float64'), side='right')
    return np.asarray(GRADES, dtype=object)[np.clip(idx, 0, len(GRADES) - 1)]


EP['risk_grade'] = pd.Series(to_grade(EP['pd']), index=EP.index, dtype='str')


def grade_table(df):
    t = df.groupby('risk_grade').agg(n=('bad', 'size'), avg_pd=('pd', 'mean'), bad_rate=('bad', 'mean'),
                                     self_cure=('self_cure30', 'mean'))
    t['share'] = t['n'] / t['n'].sum()
    return t.reindex(GRADES)[['n', 'share', 'avg_pd', 'bad_rate', 'self_cure']]


GRADE_TABLES = {s: grade_table(EP.loc[IDS[s]]) for s in ('train', 'valid', 'oot')}
GRADES_TBL = pd.concat(GRADE_TABLES, names=['sample'])
_cut_txt = ' | '.join(f'{g} <= {c:.3f}' for g, c in zip(GRADES, CUTS)) + f' | G5 > {CUTS[-1]:.3f}'
show(GRADES_TBL, f'Risk grades (PD cutpoints: {_cut_txt})')
def monotonicity(t, alpha=0.05):
    """Strict check of bad rate G1 <= ... <= G5, plus a one-sided two-proportion z-test for every inversion
    (an inversion that is not significant is sampling noise of small grades, not a broken ranking)."""
    r = t.dropna(subset=['bad_rate'])
    inv = []
    for (g1, a), (g2, b) in zip(r.iterrows(), list(r.iterrows())[1:]):
        if b['bad_rate'] < a['bad_rate']:
            p = (a['bad_rate'] * a['n'] + b['bad_rate'] * b['n']) / (a['n'] + b['n'])
            se = np.sqrt(p * (1 - p) * (1 / a['n'] + 1 / b['n']))
            pval = 0.5 * math.erfc((a['bad_rate'] - b['bad_rate']) / se / math.sqrt(2)) if se > 0 else 0.0
            inv.append((g1, g2, pval))
    strict = not inv
    signif = [x for x in inv if x[2] < alpha]
    status = ('monotone' if strict else 'SIGNIFICANT inversion' if signif else
              'inversion within sampling noise (' + ', '.join(f'{a}>{b} p={p:.2f}' for a, b, p in inv) + ')')
    return dict(strict=strict, significant_inversion=bool(signif), status=status)


MONOTONE = {}
for _s, _t in GRADE_TABLES.items():
    MONOTONE[_s] = monotonicity(_t)
    _r = _t['bad_rate'].dropna().to_numpy()
    print(f"{_s:5s}: bad rate by grade {' -> '.join(f'{x:.0%}' for x in _r)}  -> {MONOTONE[_s]['status']}"
          + ('  WARN' if MONOTONE[_s]['significant_inversion'] else ''))

with open(MODEL_DIR / 'model_bundle.pkl', 'wb') as _fh:
    pickle.dump(dict(model=MODEL, model_kind=MODEL_KIND, features=FEATURES, cat_levels=CAT_LEVELS, lr=LR,
                     lr_features=LR_FEATURES, lr_clip=None if LR_CLIP is None else (LR_CLIP[0].to_dict(), LR_CLIP[1].to_dict()),
                     grade_cutpoints=CUTS.tolist(), grades=GRADES, config=CONFIG,
                     data_cut=str(DATA_CUT.date()), run_mode=RUN_MODE), _fh)
_ = (MODEL_DIR / 'cutpoints.json').write_text(json.dumps({'grades': GRADES, 'pd_cutpoints': CUTS.tolist(),
                                                       'quantiles': CONFIG['GRADE_QUANTILES']}, indent=1), encoding='utf-8')

## 9. Segmentation
### 9.0 Regulatory guardrails (Kazakhstan, 2026; from the research memo. Legal must confirm items marked [verify])
- **Banking law.** Law 2444 (art. 36) was replaced by **Law 258-VIII of 16.01.2026**, in force from 19.03.2026. Borrower settlement is now **art. 61** [verify transition].
- **Notice.** The borrower must be notified **within 10 calendar days** of the overdue, in writing **and** via digital channels. This applies to **every segment and every challenger arm**. The output carries `notify_due_date`.
- **Change of terms.** The borrower may apply **within 30 days**, and the bank answers in writing **within 15 days** (`restructuring_window_end`). New consumer loans are banned above **30 DPD** (art. 58) [verify]. Hardship relief must therefore be a **change of terms**, never a new loan.
- **Contact limits** (Rules No. 136):
  - calls: weekdays 08:00-21:00 with **at most 3 a day**; weekends and holidays 09:00-19:00 with **at most 2 a day**;
  - only from identifiable bank numbers;
  - visits on weekdays only, **at most 1 a day and 3 a week**.
- **Third parties.** Banking secrecy applies: the debt is never disclosed to relatives or employers. `third_party_contact_flag` marks CRM "3 лицу" outcomes for compliance review.
- **Debiting.** A bank payment demand may take **at most 50%** of a balance and must leave **2 x ПМ = 101,702 KZT a month**, from 21.10.2026 (`debitable_funds`). Social, pension and alimony accounts are protected. Debiting at other banks needs contract consent [uncertain].
- **Collectors.** Hand-over only after **90 DPD**, under an agency model. Sale of retail debt to collectors is **banned until 01.05.2027**.
- **Bankruptcy** (Law 178-VII). Out-of-court bankruptcy requires debt of at most 1,600 MRP (6.92M KZT), **no property** and 12 months without payments. Car owners fail the property test, which plausibly explains why every application in `bankrupt` was refused (R03).
- **«Лудоман»** (gambling-restriction list) is a vulnerability flag: no credit offers and a monitored restructuring. **«Стоп-кредит»** has 5.1M users and is revocable. It is a *weak* signal, so it sets `no_credit_offer` but is **not** a hard rule; this is why R08 is gambling only.
- **Soft-stage collateral triggers mean inspection, not repossession.** Out-of-court sale of the car is reportedly barred below 3 months overdue [verify].

### Segment playbook
The overlay rules apply first, in priority order, and the first match wins. Everyone else is placed on the risk x capacity x engagement matrix:

| segment | definition | treatment |
|---|---|---|
| R01 deceased | `FLG_DEATH` = 1 | stop contact; legal / insurance / heirs |
| R02 bankruptcy | active bankruptcy exclusion, or PKB bankruptcy / court event | stop contact; legal process |
| R03 bankruptcy attempt | refused out-of-court application within 365 days | senior agent; documented art. 61 offer; car check |
| R04 social exclusion | active СУСН / УВЕК / упрощенка / БизБиргемиз list | soft digital reminders, early restructuring call |
| R05 enforcement | open ИР3/ИР4 orders at obs | restructuring call; auto-debit likely to fail |
| R06 abroad / skip | abroad logins >= 50% (30 days), CRM abroad note, last event abroad, or unreachable | skip-trace; field check of the car |
| R07 FPD / fraud | MOB <= 2 at entry | fraud verification; dealer review |
| R08 gambling | PKB «Лудоман» | vulnerability script; no credit |
| R09 collateral alert | PKB pledge change / assignment | car inspection within 3 days; legal review |
| S1 self-cure | G1-G2, engaged and has capacity | push + SMS + auto-debit retry; no operator before day 7 |
| S2 low-risk remind | other G1-G2 | robot / IVR + SMS on days 1-5; operator from day 6 |
| S3 standard | G3 (not hardship) | SC 4-30 operator queue; PTP tracking |
| S4 hardship | G3-G5, engaged, no capacity | proactive art. 61 restructuring / deferral |
| S5 high-risk contactable | G4-G5, engaged (capacity yes or unknown) | senior collectors from day 1; daily PTP control |
| S6 high-risk unreachable | G4-G5, not engaged | skip-trace; early field visit; car check |

- **Capacity** = `capacity_ratio >= 1.5` or `depo_cover >= 1`. It is *unknown* when both are missing.
- **Engagement** = logins in the last 30 days, or MAU flags, or `CNT_SESSIONS > 0`, or a past call contact rate of at least 50%.

**Treatment contamination.** Past outcomes reflect the current SC 4-30 calling strategy, so the backtest shows *risk under today's treatment*, not uplift. Uplift needs randomised challengers: 15% of each segment by contract hash (`arm`). The first test is S1 with *no operator until day 10*.

In [ ]:
SEGMENT_TREATMENT = {   # edit here: action, channels, first operator day, challenger idea
    'R01_DECEASED': dict(action='Stop collection contact; verify the death record; legal / insurance / heirs process',
                         channels='none (legal)', operator_from_day=None, challenger=None),
    'R02_BANKRUPTCY': dict(action='Stop collection contact; legal process (bankruptcy / court)',
                           channels='legal', operator_from_day=None, challenger=None),
    'R03_BANKRUPTCY_ATTEMPT': dict(action='Senior agent; document an art. 61 change-of-terms offer; car (collateral) check',
                                   channels='senior agent, letter', operator_from_day=1, challenger='early voluntary-sale offer'),
    'R04_SOCIAL_EXCLUSION': dict(action='Soft digital reminders or an early restructuring call; do not rely on auto-debit',
                                 channels='push, SMS, call', operator_from_day=5, challenger='restructuring offer on day 1'),
    'R05_ENFORCEMENT': dict(action='Early restructuring call; auto-debit likely to fail (bailiff orders on accounts)',
                            channels='call, letter', operator_from_day=1, challenger='payment holiday offer'),
    'R06_ABROAD_SKIPTRACE': dict(action='Skip-trace; field check of the car; contact the borrower / obligated parties only',
                                 channels='skip-trace, field', operator_from_day=1, challenger=None),
    'R07_FPD_FRAUD_CHECK': dict(action='Fraud verification (identity, income, pledge); dealer review',
                                channels='fraud unit, call', operator_from_day=1, challenger=None),
    'R08_GAMBLING': dict(action='Vulnerability script; monitored restructuring; no credit offers',
                         channels='trained agent', operator_from_day=3, challenger=None),
    'R09_COLLATERAL_ALERT': dict(action='Car inspection within 3 days; legal review',
                                 channels='field, legal', operator_from_day=1, challenger=None),
    'S1_SELF_CURE': dict(action='Homebank push + SMS, auto-debit retry within the 50% / 2xПМ limits; no operator before day 7',
                         channels='push, SMS, auto-debit', operator_from_day=7, challenger='no operator until day 10'),
    'S2_LOW_RISK_REMIND': dict(action='Robot / IVR + SMS on days 1-5; operator from day 6',
                               channels='IVR, SMS', operator_from_day=6, challenger='push instead of IVR'),
    'S3_STANDARD': dict(action='SC 4-30 operator queue; PTP tracking',
                        channels='operator', operator_from_day=4, challenger='operator from day 2'),
    'S4_HARDSHIP': dict(action='Proactive art. 61 restructuring / deferral (change of terms, not a new loan)',
                        channels='operator, Homebank offer', operator_from_day=2, challenger='digital self-service offer'),
    'S5_HIGH_RISK_CONTACTABLE': dict(action='Senior collectors from day 1; daily PTP control within the call caps',
                                     channels='senior operator', operator_from_day=1, challenger='restructuring offer on day 5'),
    'S6_HIGH_RISK_UNREACHABLE': dict(action='Skip-trace, early field visit, car check',
                                     channels='skip-trace, field', operator_from_day=1, challenger=None),
}
SEGMENT_ORDER = list(SEGMENT_TREATMENT)
RULE_TEXT = {
    'R01_DECEASED': 'FLG_DEATH = 1', 'R02_BANKRUPTCY': 'active bankruptcy exclusion / PKB bankruptcy or court',
    'R03_BANKRUPTCY_ATTEMPT': 'refused out-of-court bankruptcy <= 365d', 'R04_SOCIAL_EXCLUSION': 'active СУСН/УВЕК/упрощенка/БизБиргемиз list',
    'R05_ENFORCEMENT': 'open ИР3/ИР4 collection orders', 'R06_ABROAD_SKIPTRACE': 'abroad logins / CRM abroad note / unreachable',
    'R07_FPD_FRAUD_CHECK': 'first/second payment default (MOB <= 2)', 'R08_GAMBLING': 'PKB «Лудоман»',
    'R09_COLLATERAL_ALERT': 'PKB collateral change / assignment',
}


def rule_flags(R):
    """Overlay rules on a feature frame (missing blocks -> rule cannot fire). Column order = priority."""
    f = {
        'R01_DECEASED': col(R, 'pr_flag_FLG_DEATH').eq(1),
        'R02_BANKRUPTCY': (col(R, 'exc_active_bankruptcy').eq(1) | col(R, 'pkb_bankruptcy_all').gt(0)
                           | (col(R, 'pkb_court_all').gt(0) & CONFIG['R02_PKB_COURT'])),
        'R03_BANKRUPTCY_ATTEMPT': col(R, 'bk_flag_365d').eq(1),
        'R04_SOCIAL_EXCLUSION': col(R, 'exc_active_other_list').eq(1),
        'R05_ENFORCEMENT': col(R, 'en_n_open').gt(0),
        'R06_ABROAD_SKIPTRACE': (col(R, 'geo_share_abroad_30d').ge(0.5) | col(R, 'cl_abroad_kw').eq(1)
                                 | col(R, 'pr_last_event_abroad').eq(1)
                                 | (col(R, 'geo_logins_90d').eq(0) & col(R, 'cl_contact_rate_365d').eq(0)
                                    & col(R, 'cl_n_calls_365d').ge(3))),
        'R07_FPD_FRAUD_CHECK': col(R, 'ct_mob_at_obs').le(2),
        'R08_GAMBLING': col(R, 'pkb_ludoman_all').gt(0),
        'R09_COLLATERAL_ALERT': col(R, 'pkb_collateral_change_all').gt(0) | col(R, 'pkb_assignment_sold_all').gt(0),
    }
    return pd.DataFrame(f, index=R.index).fillna(False).astype(bool)


def capacity_axis(R):
    """1 = can pay (income / deposits cover the instalment), 0 = cannot, NaN = unknown."""
    cr, dc = col(R, 'capacity_ratio'), col(R, 'depo_cover')
    yes = cr.ge(CONFIG['CAPACITY_RATIO_MIN']) | dc.ge(CONFIG['DEPO_COVER_MIN'])
    known = cr.notna() | dc.notna()
    return pd.Series(np.where(yes, 1.0, np.where(known, 0.0, np.nan)), index=R.index)


def engaged_axis(R):
    """Reachable / willing: app logins, MAU flags, sessions or a good past contact rate."""
    return (col(R, 'geo_logins_30d').gt(0) | col(R, 'pr_flag_MAU_FIREBASE').eq(1) | col(R, 'pr_flag_MAU_AUTORIZATION').eq(1)
            | col(R, 'pr_CNT_SESSIONS').gt(0) | col(R, 'cl_contact_rate_365d').ge(CONFIG['CONTACT_RATE_ENGAGED']))


def segment_frame(R, pdv, grade):
    """Rule overlay (priority order, first match wins) + model segments S1-S6 for everyone else."""
    F = rule_flags(R)
    cap, eng = capacity_axis(R), engaged_axis(R).to_numpy()
    g = pd.to_numeric(pd.Series(grade, index=R.index).str[1:], errors='coerce').to_numpy()
    c1, c0 = cap.eq(1).to_numpy(), cap.eq(0).to_numpy()
    seg_model = np.select([(g <= 2) & eng & c1, g <= 2, (g >= 3) & c0 & eng, (g >= 4) & eng, g >= 4],
                          ['S1_SELF_CURE', 'S2_LOW_RISK_REMIND', 'S4_HARDSHIP', 'S5_HIGH_RISK_CONTACTABLE',
                           'S6_HIGH_RISK_UNREACHABLE'], default='S3_STANDARD')
    arr, names = F.to_numpy(), np.asarray(F.columns)
    any_rule = arr.any(axis=1)
    first = names[arr.argmax(axis=1)]
    cap_txt = np.where(c1, 'yes', np.where(c0, 'no', 'unknown'))
    model_reason = [f'{gr}, PD {p:.1%}, capacity {c}, engaged {"yes" if e else "no"}'
                    for gr, p, c, e in zip(grade, pdv, cap_txt, eng)]
    return pd.DataFrame({
        'segment': np.where(any_rule, first, seg_model),
        'segment_reason': np.where(any_rule, [RULE_TEXT[r] for r in first], model_reason),
        'model_segment': seg_model,
        'rules_hit': [';'.join(names[row]) for row in arr],
        'capacity': cap_txt, 'engaged': eng.astype('int8'),
    }, index=R.index).astype({'segment': 'str', 'segment_reason': 'str', 'model_segment': 'str', 'rules_hit': 'str',
                              'capacity': 'str'})


# history: everything as of obs_date
SEG_H = segment_frame(X, EP['pd'].to_numpy(), EP['risk_grade'].to_numpy())
EP = EP.drop(columns=[c for c in SEG_H.columns if c in EP.columns]).join(SEG_H)

# current stock: rules / capacity / engagement with the LATEST info (score_date), PD at entry, FPD at entry
_RN = XNOW.copy()
if 'ct_mob_at_obs' in X.columns:
    _RN['ct_mob_at_obs'] = X.loc[_RN.index, 'ct_mob_at_obs']
SEG_S = segment_frame(_RN, EP.loc[_RN.index, 'pd'].to_numpy(), EP.loc[_RN.index, 'risk_grade'].to_numpy())
print('rule overlay hit rates (history at obs | stock now):')
display(pd.DataFrame({'history': rule_flags(X).mean(), 'stock_now': rule_flags(_RN).mean() if len(_RN) else np.nan}))

In [ ]:
_cc = CONFIG['COMPLIANCE']
S = STOCK.join(SEG_S).join(contracts.set_index('contract_number')[['process_ref', 'iin']], on='contract_number')
S = S.assign(pd=EP.loc[S.index, 'pd'], risk_grade=EP.loc[S.index, 'risk_grade'],
             treatment=S['segment'].map({k: v['action'] for k, v in SEGMENT_TREATMENT.items()}),
             channels=S['segment'].map({k: v['channels'] for k, v in SEGMENT_TREATMENT.items()}),
             operator_from_day=S['segment'].map({k: v['operator_from_day'] for k, v in SEGMENT_TREATMENT.items()}),
             score_date=SCORE_DATE,
             notify_due_date=(S['entry_date'] + pd.Timedelta(days=_cc['NOTIFY_DAYS'])).astype('datetime64[us]'),
             restructuring_window_end=(S['entry_date'] + pd.Timedelta(days=_cc['RESTRUCT_APPLY_DAYS'])).astype('datetime64[us]'),
             capacity_ratio=col(XNOW, 'capacity_ratio').reindex(S.index), depo_cover=col(XNOW, 'depo_cover').reindex(S.index),
             debitable_funds=col(XNOW, 'debitable_funds').reindex(S.index),
             logins_30d=col(XNOW, 'geo_logins_30d').reindex(S.index),
             contact_rate_365d=col(XNOW, 'cl_contact_rate_365d').reindex(S.index),
             enforcement_open=col(XNOW, 'en_n_open').reindex(S.index),
             no_credit_offer=(col(XNOW, 'pkb_stop_credit_all').gt(0) | col(XNOW, 'pkb_ludoman_all').gt(0)).reindex(S.index)
                             .fillna(False).astype('int8') | (S['dpd_now'] > _cc['NO_NEW_LOAN_DPD']).astype('int8'),
             third_party_contact_flag=col(XNOW, 'cl_n_third_party_365d').gt(0).reindex(S.index).fillna(False).astype('int8'),
             arm=np.where(hash_share(S['contract_number']) < CONFIG['CHALLENGER_SHARE'], 'challenger', 'champion'),
             top_drivers=top_drivers(XE.loc[S.index]))
S['notify_days_left'] = (S['notify_due_date'] - SCORE_DATE).dt.days
S['challenger_treatment'] = np.where(S['arm'] == 'challenger',
                                     S['segment'].map({k: v['challenger'] for k, v in SEGMENT_TREATMENT.items()}), None)

SEG_CURRENT = (S.groupby('segment').agg(n=('pd', 'size'), avg_pd=('pd', 'mean'), avg_dpd_now=('dpd_now', 'mean'),
                                        challenger=('arm', lambda a: int((a == 'challenger').sum())))
                .reindex([s for s in SEGMENT_ORDER if s in set(S['segment'])]))
SEG_CURRENT['share'] = SEG_CURRENT['n'] / max(len(S), 1)
SEG_CURRENT['treatment'] = SEG_CURRENT.index.map({k: v['action'] for k, v in SEGMENT_TREATMENT.items()})
show(SEG_CURRENT, f'Current stock ({len(S)} contracts in DPD {CONFIG["SOFT_DPD_MIN"]}-{CONFIG["SOFT_DPD_MAX"]} on {DATA_CUT:%Y-%m-%d})')
show(S[['contract_number', 'dpd_now', 'pd', 'risk_grade', 'segment', 'arm', 'notify_due_date', 'top_drivers']]
     .sort_values('pd', ascending=False).head(10), 'Highest-PD contracts in the stock (IINs are in the output files)')

### 9.3 Backtest on labelled history
The segments are applied to past episodes as of their `obs_date`. Bad rates are the outcomes under the treatment of the time (see the contamination caveat above). A good segmentation shows a clear spread: S1/S2 should have a low bad rate and high self-cure, S5/S6 a high bad rate. The axis check under the table validates the engagement and capacity splits inside each risk grade.

In [ ]:
def seg_table(df):
    t = df.groupby('segment').agg(n=('bad', 'size'), avg_pd=('pd', 'mean'), bad90=('bad', 'mean'),
                                  self_cure=('self_cure30', 'mean'), avg_max_dpd=('max_dpd', 'mean'))
    t['roll31_60d'] = df[df['labeled_roll']].groupby('segment')['roll31_60d'].mean()
    t['share'] = t['n'] / t['n'].sum()
    return t.reindex([s for s in SEGMENT_ORDER if s in t.index])[['n', 'share', 'avg_pd', 'bad90', 'self_cure',
                                                                   'roll31_60d', 'avg_max_dpd']]


BACKTEST = pd.concat({'all_labeled': seg_table(EP[EP['labeled']]), 'oot': seg_table(EP.loc[IDS['oot']])},
                     names=['sample'])
show(BACKTEST, 'Segment backtest (bad90 = actual 90+ within 6 months)')
_lab = EP[EP['labeled'] & (EP['rules_hit'] == '')]
AXIS_CHECK = _lab.pivot_table(index='risk_grade', columns='engaged', values='bad', aggfunc='mean', observed=True)
AXIS_CHECK.columns = [f'bad90 | engaged={c}' for c in AXIS_CHECK.columns]
AXIS_CHECK['bad90 | capacity=yes'] = _lab[_lab['capacity'] == 'yes'].groupby('risk_grade')['bad'].mean()
AXIS_CHECK['bad90 | capacity=no'] = _lab[_lab['capacity'] == 'no'].groupby('risk_grade')['bad'].mean()
show(AXIS_CHECK.reindex(GRADES), 'Axis check: bad rate by grade x engagement / capacity (labelled, no rule hit)')

### 9.4 Behavioural clusters (profiling)
About 15 behavioural features go through median imputation and a normal `QuantileTransformer`, then KMeans with k = 4..8 chosen by silhouette on a sample of at most 20k. Each cluster is profiled and auto-named after its three largest z-score deviations. The cross-tab with the segments shows how the risk segments map onto behaviour. This is descriptive, not a second model.

In [ ]:
_CL_SPEC = {  # cluster feature -> (column in X, transform)
    'pd': (None, None), 'prior_episodes': ('dh_n_prior_episodes', None), 'log_income': ('pr_income_proxy', 'log'),
    'log_depo': ('pr_DEPO_SUM', 'log'), 'pkb_overdue': ('pr_SUM_PKB_OVRD_ACT', 'log'), 'logins_30d': ('geo_logins_30d', None),
    'enforcement_open': ('en_n_open', None), 'mob': ('ct_mob_at_obs', None), 'log_amount': ('ct_log_amount', None),
    'age': ('pr_AGE', None), 'digital_ratio': ('pr_DIGITAL_RATIO', None), 'contact_rate': ('cl_contact_rate_365d', None),
    'rejections_90d': ('rej_n_90d', None), 'capacity_ratio': ('capacity_ratio', 'log'), 'calls_365d': ('cl_n_calls_365d', None),
}
_cl = {}
for _k, (_c, _tf) in _CL_SPEC.items():
    if _c is None:
        _cl[_k] = EP['pd']
    elif _c in X.columns:
        _v = X[_c].astype('float64')
        _cl[_k] = np.log1p(_v.clip(lower=0)) if _tf == 'log' else _v
CL = pd.DataFrame(_cl, index=EP.index)
_miss = CL.isna().mean()
CL = CL.loc[:, _miss <= 0.6]
print(f"cluster features ({CL.shape[1]}): {list(CL.columns)}; dropped (>60% missing): {list(_miss[_miss > 0.6].index)}")

CLUSTER_PROFILE = None
if CL.shape[1] >= 2 and len(CL) >= 50:
    _Z = make_pipeline(SimpleImputer(strategy='median'),
                       QuantileTransformer(output_distribution='normal', n_quantiles=min(1000, len(CL)),
                                           random_state=CONFIG['RANDOM_STATE'])).fit_transform(CL)
    _rng = np.random.default_rng(CONFIG['RANDOM_STATE'])
    _smp = _rng.choice(len(_Z), size=min(len(_Z), CONFIG['CLUSTER_SAMPLE']), replace=False)
    _sil = {}
    for _k in range(CONFIG['CLUSTER_K_RANGE'][0], CONFIG['CLUSTER_K_RANGE'][1] + 1):
        _km = KMeans(n_clusters=_k, n_init=10, random_state=CONFIG['RANDOM_STATE']).fit(_Z[_smp])
        _sil[_k] = silhouette_score(_Z[_smp], _km.labels_, sample_size=min(len(_smp), 5000),
                                    random_state=CONFIG['RANDOM_STATE'])
    K_BEST = max(_sil, key=_sil.get)
    KM = KMeans(n_clusters=K_BEST, n_init=10, random_state=CONFIG['RANDOM_STATE']).fit(_Z[_smp])
    _lab_cl = KM.predict(_Z)
    _zc = pd.DataFrame(_Z, index=EP.index, columns=CL.columns).groupby(_lab_cl).mean()
    _zc = (_zc - _Z.mean(axis=0)) / _Z.std(axis=0)
    _names = {i: ' / '.join(f"{'high' if _zc.loc[i, f] > 0 else 'low'} {f}"
                            for f in _zc.loc[i].abs().sort_values(ascending=False).index[:3]) for i in _zc.index}
    EP['cluster'] = pd.Series([f'C{i}: {_names[i]}' for i in _lab_cl], index=EP.index, dtype='str')
    print('silhouette by k: ' + ', '.join(f'{k}: {v:.3f}' for k, v in _sil.items()) + f' -> k = {K_BEST}')
    _labd = EP[EP['labeled']]
    CLUSTER_PROFILE = pd.concat([
        EP.groupby('cluster').size().rename('n'),
        (EP.groupby('cluster').size() / len(EP)).rename('share'),
        _labd.groupby('cluster')['bad'].mean().rename('bad90'),
        _labd.groupby('cluster')['self_cure30'].mean().rename('self_cure'),
        CL.groupby(EP['cluster']).median().add_prefix('median_'),
    ], axis=1)
    show(CLUSTER_PROFILE, 'Behavioural clusters (bad90 / self-cure on labelled episodes)')
    show(pd.crosstab(EP['cluster'], EP['segment']).reindex(columns=[s for s in SEGMENT_ORDER if s in set(EP['segment'])]),
         'Clusters x segments (all episodes)')
    S['cluster'] = EP.loc[S.index, 'cluster']
else:
    EP['cluster'] = pd.Series(np.nan, index=EP.index, dtype='str')
    S['cluster'] = pd.Series(np.nan, index=S.index, dtype='str')
    print('clustering skipped (too few features / episodes)')

### 9.5 Dealer FPD table and 9.6 monthly monitoring
- **Dealer table.** First-payment default is a lagging dealer-fraud signal (vendor data: more than 70% of early defaults show origination fraud). The table shows the FPD rate per `partner_name`, with a Wilson 95% confidence interval, for dealers with at least 30 contracts seasoned 3+ months, plus the bad90 rate of their labelled episodes.
- **Monitoring.** Each month: score PSI against train (0.10 = investigate, 0.25 = act), the mature-cohort Gini against the valid Gini (a relative drop of 10% = amber, 20% = red), the roll31 early proxy, and the grade mix.

In [ ]:
_fpd_contracts = set(EP.loc[col(X, 'dh_fpd_flag').eq(1).to_numpy(), 'contract_number'])
_dl = contracts[contracts['open_date'] <= DATA_CUT - pd.DateOffset(months=3)]
_dl = _dl.assign(dealer=as_cat(col(_dl, 'partner_name'), upper=False).fillna('<missing>'),
                 fpd=_dl['contract_number'].isin(_fpd_contracts).astype(int))
DEALERS = _dl.groupby('dealer').agg(n_contracts=('fpd', 'size'), n_fpd=('fpd', 'sum'))
DEALERS['fpd_rate'] = DEALERS['n_fpd'] / DEALERS['n_contracts']
DEALERS['fpd_ci_low'], DEALERS['fpd_ci_high'] = wilson(DEALERS['n_fpd'], DEALERS['n_contracts'])
_le = EP[EP['labeled']].join(_dl.set_index('contract_number')['dealer'], on='contract_number')
_b = _le.groupby('dealer').agg(n_labeled_eps=('bad', 'size'), n_bad=('bad', 'sum'))
DEALERS = DEALERS.join(_b)
DEALERS['bad90_rate'] = DEALERS['n_bad'] / DEALERS['n_labeled_eps']
DEALERS['bad90_ci_low'], DEALERS['bad90_ci_high'] = wilson(DEALERS['n_bad'], DEALERS['n_labeled_eps'])
DEALERS = DEALERS[DEALERS['n_contracts'] >= 30].sort_values('fpd_rate', ascending=False)
_port_fpd = _dl['fpd'].mean()
DEALERS['flag'] = np.where(DEALERS['fpd_ci_low'] > _port_fpd, 'FPD above portfolio (CI)', '')
show(DEALERS, f'Dealers: FPD rate (portfolio {_port_fpd:.1%}) and bad90 of labelled episodes (n >= 30 contracts)')

In [ ]:
_ref_gini = GINI.get((MAIN, 'valid'), np.nan)          # out-of-sample reference Gini of the main model
_base = EP.loc[IDS['train'], 'pd']
_rows = []
for _m, _d in EP.groupby('obs_month'):
    _r = dict(obs_month=str(_m), n=len(_d), mean_pd=_d['pd'].mean(), psi_vs_train=psi(_base, _d['pd']),
              share_G4_G5=_d['risk_grade'].isin(['G4', 'G5']).mean())
    _l = _d[_d['labeled']]
    _r['labeled'] = len(_l)
    _r['bad_rate'] = _l['bad'].mean() if len(_l) else np.nan
    _r['gini'] = perf(_l['bad'], _l['pd'])['gini'] if _l['bad'].sum() >= 20 and (1 - _l['bad']).sum() >= 20 else np.nan
    _lr = _d[_d['labeled_roll']]
    _r['roll31_rate'] = _lr['roll31_60d'].mean() if len(_lr) else np.nan
    _rows.append(_r)
MONITORING = pd.DataFrame(_rows).set_index('obs_month')
MONITORING['psi_status'] = np.select([MONITORING['n'] < 100, MONITORING['psi_vs_train'] >= 0.25,
                                      MONITORING['psi_vs_train'] >= 0.10], ['low n', 'ACT', 'INVESTIGATE'], default='ok')
_drop_rel = 1 - MONITORING['gini'] / _ref_gini
MONITORING['gini_status'] = np.select([_drop_rel > 0.20, _drop_rel > 0.10, MONITORING['gini'].notna()],
                                      ['RED', 'AMBER', 'ok'], default='n/a')
show(MONITORING.tail(12), f'Monthly monitoring, last 12 months (reference: valid Gini {_ref_gini:.3f}, train PD baseline; '
                          'Gini needs >= 20 bads and goods)')

## 10. Charts
There are five charts, saved to `out/charts/*.png`. Each has one y-axis, a legend only where there are two series, and value labels only where they help.

In [ ]:
PAL = dict(s1='#2a78d6', s2='#eb6834', s3='#1baf7a', ink='#0b0b0b', ink2='#52514e', grid='#e4e3df', surface='#ffffff')
plt.rcParams.update({
    'figure.facecolor': PAL['surface'], 'axes.facecolor': PAL['surface'], 'savefig.facecolor': PAL['surface'],
    'axes.edgecolor': PAL['grid'], 'axes.linewidth': 0.8, 'axes.labelcolor': PAL['ink2'], 'axes.titlecolor': PAL['ink'],
    'axes.titlesize': 11, 'axes.titlelocation': 'left', 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'axes.axisbelow': True, 'grid.color': PAL['grid'], 'grid.linewidth': 0.6,
    'xtick.color': PAL['ink2'], 'ytick.color': PAL['ink2'], 'xtick.labelsize': 8.5, 'ytick.labelsize': 8.5,
    'text.color': PAL['ink'], 'font.size': 9, 'legend.frameon': False, 'legend.fontsize': 8.5, 'figure.dpi': 100,
})
CHARTS = {}


def _save(fig, name):
    fig.tight_layout()
    path = OUT / 'charts' / f'{name}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight')
    CHARTS[name] = path
    plt.show()
    plt.close(fig)


def _label_bars(ax, bars, fmt, horizontal=False, extra=None):
    for i, b in enumerate(bars):
        v = b.get_width() if horizontal else b.get_height()
        if not np.isfinite(v):
            continue
        xy = (v, b.get_y() + b.get_height() / 2) if horizontal else (b.get_x() + b.get_width() / 2, v)
        ax.annotate(fmt(v) + (extra[i] if extra is not None else ''), xy, xytext=(3, 0) if horizontal else (0, 2), textcoords='offset points',
                    ha='left' if horizontal else 'center', va='center' if horizontal else 'bottom',
                    color=PAL['ink2'], fontsize=8)


# (1) actual bad rate by risk grade: train vs OOT
fig, ax = plt.subplots(figsize=(7, 3.6))
_x = np.arange(len(GRADES))
for _i, (_s, _c, _lbl) in enumerate((('train', PAL['s1'], 'train (out-of-fold PD)'), ('oot', PAL['s2'], 'out-of-time'))):
    _v = GRADE_TABLES[_s]['bad_rate'].to_numpy(dtype='float64')
    _bars = ax.bar(_x + (_i - 0.5) * 0.36, _v, width=0.34, color=_c, label=_lbl)
    _label_bars(ax, _bars, lambda v: f'{v:.0%}')
ax.set_xticks(_x, GRADES)
ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
ax.grid(axis='x', visible=False)
ax.set_title('Actual 90+ DPD rate within 6 months, by risk grade')
ax.legend(loc='upper left')
_save(fig, '1_bad_rate_by_grade')

# (2) segment size - current stock (falls back to history if the stock is empty)
_sz = S['segment'].value_counts() if len(S) else EP['segment'].value_counts()
_sz = _sz.reindex([s for s in SEGMENT_ORDER if s in _sz.index])[::-1]
fig, ax = plt.subplots(figsize=(7, 0.32 * len(_sz) + 1.2))
_bars = ax.barh(_sz.index, _sz.to_numpy(), height=0.6, color=PAL['s1'])
_label_bars(ax, _bars, lambda v: f'{v:,.0f}', horizontal=True)
ax.grid(axis='y', visible=False)
ax.set_title(f"Segment size: {'current stock' if len(S) else 'all episodes'} (contracts)")
_save(fig, '2_segment_size')

# (3) bad rate by segment - labelled history
_bt = BACKTEST.loc['all_labeled'][::-1]
_br = _bt['bad90']
fig, ax = plt.subplots(figsize=(7, 0.32 * len(_br) + 1.2))
_bars = ax.barh(_br.index, _br.to_numpy(dtype='float64'), height=0.6, color=PAL['s1'])
_label_bars(ax, _bars, lambda v: f'{v:.0%}', horizontal=True, extra=[f'  n={n:,.0f}' for n in _bt['n']])
ax.set_xlim(0, min(1.0, float(np.nanmax(_br.to_numpy(dtype='float64')))) * 1.18 + 0.02)
ax.set_xticks([t for t in ax.get_xticks() if 0 <= t <= 1.0 + 1e-9])
ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
ax.grid(axis='y', visible=False)
ax.set_title('Actual 90+ DPD rate by segment (labelled history)')
_save(fig, '3_bad_rate_by_segment')

# (4) top-20 feature importance
_imp = IMP_TABLE['gain_share'].head(20)[::-1]
fig, ax = plt.subplots(figsize=(7, 0.28 * len(_imp) + 1.2))
ax.barh(_imp.index, _imp.to_numpy(), height=0.6, color=PAL['s1'])
ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
ax.grid(axis='y', visible=False)
ax.set_title(f'Top-20 features: share of total gain ({MODEL_KIND})')
_save(fig, '4_feature_importance')

# (5) monthly entries and bad rate - two small multiples, one axis each
_mt = MONTHLY.copy()
_mt.index = _mt.index.to_timestamp()
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 4.8), sharex=True)
ax1.bar(_mt.index, _mt['entries'], width=20, color=PAL['s1'])
ax1.grid(axis='x', visible=False)
ax1.set_title('Entries into delinquency per month')
_lb = _mt['bad_rate'].dropna()
ax2.plot(_lb.index, _lb.to_numpy(), color=PAL['s1'], linewidth=2, solid_capstyle='round')
ax2.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
ax2.set_ylim(bottom=0)
ax2.set_title('Bad rate of labelled entries (90+ DPD within 6 months)')
_save(fig, '5_monthly_entries_bad_rate')

## 11. Outputs
All files go to `OUT_DIR` (`out/`, which git ignores; the files contain IINs):
- `segments_current_stock.parquet` + `.xlsx`: one row per stock contract, with PD, grade, segment and reason, treatment, challenger arm, compliance dates, key driver flags and top SHAP drivers.
- `segments_history.parquet`: labelled episodes with PD, grade, segment, cluster and the outcomes.
- `segment_report.xlsx`: data_quality, block_coverage, model_metrics, grades, segments_backtest, segments_current, clusters, feature_importance, psi_dropped, plus monthly, dealers, monitoring and treatments.
- `model/`: the LightGBM text model, the pickled bundle (model, LR, features, category levels, cutpoints, config), and features / config / cutpoints JSON.
- `charts/*.png`, `run_summary.json`.

In [ ]:
STOCK_COLS = ['contract_number', 'process_ref', 'iin', 'entry_date', 'dpd_now', 'dpd_asof', 'score_date', 'pd',
              'risk_grade', 'segment', 'segment_reason', 'treatment', 'channels', 'operator_from_day', 'arm',
              'challenger_treatment', 'rules_hit', 'model_segment', 'capacity', 'engaged', 'capacity_ratio',
              'depo_cover', 'debitable_funds', 'logins_30d', 'contact_rate_365d', 'enforcement_open',
              'no_credit_offer', 'third_party_contact_flag', 'notify_due_date', 'notify_days_left',
              'restructuring_window_end', 'cluster', 'top_drivers']
STOCK_OUT = S.reset_index()[['episode_id'] + [c for c in STOCK_COLS if c in S.columns]].sort_values('pd', ascending=False)
HIST_COLS = ['contract_number', 'entry_date', 'obs_date', 'obs_month', 'sample', 'pd', 'pd_lr', 'risk_grade',
             'segment', 'model_segment', 'rules_hit', 'capacity', 'engaged', 'cluster', 'bad', 'self_cure30',
             'roll31_60d', 'labeled_roll', 'max_dpd', 'n_months', 'reaged']
HIST_OUT = (EP[EP['labeled']].reset_index()[['episode_id'] + [c for c in HIST_COLS if c in EP.columns]]
            .merge(contracts[['contract_number', 'iin', 'process_ref']], on='contract_number', how='left')
            .assign(obs_month=lambda d: d['obs_month'].astype('str')))

STOCK_OUT.to_parquet(OUT / 'segments_current_stock.parquet', index=False)
HIST_OUT.to_parquet(OUT / 'segments_history.parquet', index=False)


def _xl(df, index=True):
    """Excel-safe copy: index as columns, periods / categories as text."""
    d = df.reset_index() if index else df.copy()
    for c in d.columns:
        if isinstance(d[c].dtype, (pd.PeriodDtype, pd.CategoricalDtype)):
            d[c] = d[c].astype('str')
    return d


def write_excel(path, sheets):
    try:
        with pd.ExcelWriter(path, engine='openpyxl') as xw:
            for name, (df, idx) in sheets.items():
                if df is not None:
                    _xl(df, idx).to_excel(xw, sheet_name=name[:31], index=False)
        log(f'written {path}')
    except Exception as e:      # Excel is a convenience copy; parquet files are the primary output
        print(f'WARN  Excel export {path} failed ({type(e).__name__}: {e})')


write_excel(OUT / 'segments_current_stock.xlsx', {'current_stock': (STOCK_OUT, False)})
TREATMENTS = pd.DataFrame(SEGMENT_TREATMENT).T.rename_axis('segment')
write_excel(OUT / 'segment_report.xlsx', {
    'data_quality': (DQ_REPORT, True), 'block_coverage': (BLOCK_COVERAGE, True), 'model_metrics': (METRICS, False),
    'grades': (GRADES_TBL, True), 'segments_backtest': (BACKTEST, True), 'segments_current': (SEG_CURRENT, True),
    'clusters': (CLUSTER_PROFILE, True), 'feature_importance': (IMP_TABLE, True), 'psi_dropped': (DROPPED, False),
    'monthly': (MONTHLY, True), 'dealers': (DEALERS, True), 'monitoring': (MONITORING, True),
    'treatments': (TREATMENTS, True), 'load_log': (pd.DataFrame(LOAD_LOG), False),
})

_gini = METRICS.set_index(['model', 'sample'])['gini']
RUN_SUMMARY = dict(
    run_mode=RUN_MODE, data_cut=str(DATA_CUT.date()), score_date=str(SCORE_DATE.date()),
    runtime_seconds=round(time.perf_counter() - T0, 1), model=MODEL_KIND,
    episodes=int(len(EP)), labeled=int(EP['labeled'].sum()), bad_rate_labeled=round(float(EP.loc[EP['labeled'], 'bad'].mean()), 4),
    n_features=len(FEATURES), model_blocks=MODEL_BLOCKS_USED,
    gini={f'{m}/{s}': round(float(v), 4) for (m, s), v in _gini.items()}, score_psi_train_oot=round(SCORE_PSI, 4),
    grade_monotone=MONOTONE, pd_cutpoints=[round(float(c), 4) for c in CUTS],
    grades_oot=GRADE_TABLES['oot'][['n', 'avg_pd', 'bad_rate']].round(4).reset_index().to_dict('records'),
    segments_stock=S['segment'].value_counts().to_dict(),
    segments_backtest=BACKTEST.loc['all_labeled', ['n', 'bad90', 'self_cure']].round(4).reset_index().to_dict('records'),
    stock_size=int(len(S)), dropped_tables=CONFIG['DROP_TABLES'],
    outputs=sorted(str(p.relative_to(OUT)) for p in OUT.rglob('*') if p.is_file()),
)
_ = (OUT / 'run_summary.json').write_text(json.dumps(RUN_SUMMARY, ensure_ascii=False, indent=1, default=str), encoding='utf-8')
show(pd.DataFrame([(str(p.relative_to(OUT)), f'{p.stat().st_size / 1024:,.0f} KB') for p in sorted(OUT.rglob('*')) if p.is_file()],
                  columns=['file', 'size']).set_index('file'), f'Files in {OUT.resolve()}')

## 12. Next steps and enrichment
- **Add a source.** Load it under a table name, add its columns and keys to `TABLE_COLUMNS` / `TABLE_KEYS`, and write a preparation step that normalises keys and parses dates. Then write `fb_new(obs, ctx)` using `events_before(...)`, `n_within(...)` and `finish(...)`, and register it in `BLOCKS` with `role='auto'`. The `MODEL_BLOCKS='auto'` rule decides whether its history is long enough for the model.
- **PKB monitoring.** It has been accumulating since 2026-09-07. Run `sql/hes_pkb_monitoring_agg.sql` monthly for a series of snapshot dates (month-ends) and store each snapshot. Once 6 months of labelled entries have snapshots before their `obs_date`, switch the block from `overlay` to `auto` and give it an as-of join on the snapshot date. The same applies to geo, vnpt and vneshpt as their history grows.
- **Monitoring** (section 9.6): run it monthly. Score PSI above 0.10 means investigate and above 0.25 means act. Also track per-feature CSI for the top-20 features. Gini on matured cohorts (6 months) is amber at a 10% relative drop and red at 20%. Watch roll31_60d as the early proxy, grade monotonicity and the promise-kept rate.
- **Champion-challenger.** Each segment routes 15% of contracts by contract hash (`arm`) to the challenger treatment in `SEGMENT_TREATMENT`. Run 2-3 cycles and compare cure, roll and cost; promote per segment. The first test is S1 with no operator until day 10. This is also what breaks the treatment contamination in the history.
- **Retrain** quarterly, or when PSI or Gini goes red. Keep the time-split OOT. Re-derive the grade cutpoints and re-check monotonicity.
- **Data asks.**
  - `contr` including closed, written-off and sold contracts, to remove survivorship bias.
  - The real ovrd semantics for the last month (is `act_date` the cut date for every row?).
  - The IIN and timestamp column names in `dds.hes_pkb_monitoring_event`.
  - Daily DPD instead of monthly maxima, for re-scoring on day 5 or 10.

## 13. Appendix: Vertica SQL for the PKB monitoring aggregate
This is the same text as `sql/hes_pkb_monitoring_agg.sql`. It produces long format, one row per (iin, rule), with 7/30/60-day and total counts as of `:snapshot_date`. **The IIN and event-timestamp column names are placeholders** (`*__PLACEHOLDER`); replace them before running. The next cell runs it through your existing `query()` helper and saves the result to parquet. It is guarded by `RUN_PKB_SQL = False`.

In [ ]:
PKB_SQL = r"""
/* =====================================================================================================
   hes_pkb_monitoring_agg.sql  (Vertica)

   PKB (Первое кредитное бюро) monitoring events -> per client (IIN) x rule aggregates, LONG format.

   Source : dds.hes_pkb_monitoring_event   (~110M rows, ~1.9M IINs, events since 2026-09-07)
   Output : one row per (iin, rule_key, rule_name):
              iin            12-char IIN, leading zeros kept
              rule_key       e.g. 'CustomRule.CMC'
              rule_name      e.g. 'Мониторинг статуса «Лудоман»' (the notebook maps names -> codes)
              cnt_7d         events in [snapshot_date - 7 days,  snapshot_date)
              cnt_30d        events in [snapshot_date - 30 days, snapshot_date)
              cnt_60d        events in [snapshot_date - 60 days, snapshot_date)
              cnt_all        all events before snapshot_date (scan bounded to 400 days)
              first_ts       first event time
              last_ts        last event time
              snapshot_date  the parameter (the aggregate is "as of" this date, no look-ahead)
   Use    : save the result as parquet  ->  CONFIG['PKB_AGG_FILE'] (default DATA_DIR/hes_pkb_agg.parquet).
            The notebook block fb_pkb_monitoring uses it for OVERLAY rules only (history is too short to
            train on); once 6+ months have accumulated, build monthly snapshots and move it into the model.

   Parameter
     :snapshot_date   DATE literal, e.g. '2026-11-18'. Only events strictly BEFORE this date are counted.
                      Python: sql.replace(':snapshot_date', "'2026-11-18'")
                      vsql  : \set snapshot_date '''2026-11-18'''

   >>> PLACEHOLDERS: the IIN and event-timestamp column names were not visible in the screenshots. <<<
     iin_col__PLACEHOLDER       -> the client IIN column        (check: SELECT * FROM dds.hes_pkb_monitoring_event LIMIT 5;)
     event_ts_col__PLACEHOLDER  -> the event date-time column   (the one spanning 2026-09-07 .. today)
   Replace every occurrence (search for "__PLACEHOLDER") before running.
   ===================================================================================================== */
WITH params AS (
    SELECT CAST(:snapshot_date AS TIMESTAMP) AS snap_ts
),
ev AS (
    SELECT
        -- PLACEHOLDER column: client IIN -> digits only, left-padded to 12 characters
        LPAD(REGEXP_REPLACE(CAST(e.iin_col__PLACEHOLDER AS VARCHAR(32)), '[^0-9]', ''), 12, '0') AS iin,
        -- PLACEHOLDER column: event timestamp
        CAST(e.event_ts_col__PLACEHOLDER AS TIMESTAMP)                                         AS event_ts,
        e.rule_key,
        e.rule_name
    FROM dds.hes_pkb_monitoring_event e
    CROSS JOIN params p
    WHERE e.event_ts_col__PLACEHOLDER <  p.snap_ts                         -- PLACEHOLDER column: strictly before the snapshot
      AND e.event_ts_col__PLACEHOLDER >= p.snap_ts - INTERVAL '400 days'   -- PLACEHOLDER column: bounded scan
      AND e.rule_name NOT LIKE 'Отписка%'                                  -- technical rule (unsubscribe), not a client event
      -- Optional: keep only the auto-loan portfolio to shrink the output
      -- AND e.iin_col__PLACEHOLDER IN (SELECT iin FROM <your_portfolio_table>)
)
SELECT
    ev.iin,
    ev.rule_key,
    ev.rule_name,
    SUM(CASE WHEN ev.event_ts >= p.snap_ts - INTERVAL '7 days'  THEN 1 ELSE 0 END) AS cnt_7d,
    SUM(CASE WHEN ev.event_ts >= p.snap_ts - INTERVAL '30 days' THEN 1 ELSE 0 END) AS cnt_30d,
    SUM(CASE WHEN ev.event_ts >= p.snap_ts - INTERVAL '60 days' THEN 1 ELSE 0 END) AS cnt_60d,
    COUNT(*)                                                                      AS cnt_all,
    MIN(ev.event_ts)                                                              AS first_ts,
    MAX(ev.event_ts)                                                              AS last_ts,
    CAST(p.snap_ts AS DATE)                                                       AS snapshot_date
FROM ev
CROSS JOIN params p
GROUP BY ev.iin, ev.rule_key, ev.rule_name, p.snap_ts
;
"""
print(PKB_SQL[:600] + "\n...")

In [ ]:
RUN_PKB_SQL = False     # set True in your Vertica-connected kernel (needs your query() helper)
if RUN_PKB_SQL:
    _snap = pd.Timestamp.today().normalize()
    _sql = PKB_SQL.replace(':snapshot_date', f"'{_snap:%Y-%m-%d}'")
    assert '__PLACEHOLDER' not in _sql, 'replace the placeholder column names in the SQL first'
    pkb_agg_df = query(_sql)                                  # noqa: F821 - your existing helper -> DataFrame
    _path = Path(CONFIG['DATA_DIR']) / CONFIG['PKB_AGG_FILE']
    pkb_agg_df.to_parquet(_path, index=False)
    print(f'saved {pkb_agg_df.shape} -> {_path}')
else:
    print('RUN_PKB_SQL = False -> skipped (set True in the Vertica kernel after replacing the placeholders)')

In [ ]:
print(json.dumps({k: RUN_SUMMARY[k] for k in ('run_mode', 'data_cut', 'episodes', 'labeled', 'bad_rate_labeled',
                                               'model', 'gini', 'grade_monotone', 'stock_size', 'segments_stock')},
                 ensure_ascii=False, indent=1, default=str))
log('done')